<a href="https://colab.research.google.com/github/matiasbasm/retail_multitienda_grupo_12/blob/master/E1_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multitienda del Sur SpA — Entrega 1
**IIC 3752 · Tecnologías para Inteligencia de Negocios · Proyecto del semestre · Grupo 12**

**Pregunta de negocio (README del caso, pregunta 1):** ¿La cadena está reteniendo a sus clientes o los está reponiendo con compradores de una sola vez?

Este notebook es la cadena de datos completa de la Entrega 1. Se ejecuta de cero con *Entorno de ejecución → Ejecutar todo*.

**Cómo está organizado.** Cada bloque de código tiene arriba su **objetivo** y termina imprimiendo un **RESULTADO** calculado en ese momento. Ninguna cifra está escrita a mano en el texto: si cambian los datos, cambian los resultados y el notebook sigue funcionando.

**Principio de la limpieza.** La sección 2 *detecta* problemas de calidad con chequeos genéricos (formatos mixtos, claves duplicadas, referencias huérfanas, reglas de negocio, valores fuera de rango, variantes de texto). La sección 3 *decide* qué hacer con cada problema, y cada decisión queda como un parámetro visible con la evidencia que la respalda. Detectar y decidir están separados a propósito.

| Sección | Contenido | Componente del informe |
|---|---|---|
| 0 | Configuración y parámetros | B |
| 1 | Extracción de las cuatro fuentes, con respaldo de la API | B |
| 2 | Detección de datos sucios (sin modificar nada) | B, F |
| 3 | Decisiones de limpieza y registro | B, F |
| 4 | Modelo dimensional | C |
| 5 | Métrica y sensibilidad a las definiciones | D |
| 6 | Caso de negocio en pesos reales | E |
| 7 | Cobertura de datos | F |
| 8 | Visualizaciones y diagrama del modelo | C, E |
| 9 | Resumen de cifras del informe | todos |

**Qué queda en el repositorio y qué no.** El repositorio guarda el código y los datos crudos (`data/raw/`). Todo lo que genera el notebook (`data/processed/`, `outputs/`) se recrea en cada ejecución y no se sube. La única excepción es el respaldo de la API del IPC, que se guarda en `data/raw/` para poder correr sin conexión (ver 1.4).

## 0. Configuración
**Objetivo:** dejar el entorno listo. Si el notebook se abre en Colab, clona el repositorio para tener los datos crudos; si se ejecuta dentro del repositorio, no clona nada.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/matiasbasm/retail_multitienda_grupo_12.git"
REPO_DIR = "retail_multitienda_grupo_12"

if not os.path.exists("data/raw/origen_pos.sqlite"):
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

EN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
print("RESULTADO: directorio de trabajo =", os.getcwd(), "| ejecutando en Colab:", EN_COLAB)

**Objetivo:** importar librerías y fijar los parámetros del análisis. Todo lo que es una decisión del grupo está aquí o en la sección 3, nunca escondido dentro de un cálculo.

In [ ]:
import csv, io, json, re, sqlite3, time, unicodedata, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import requests

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 80)
T0 = time.time()

# ---------------- Parámetros del análisis ----------------
GRUPO = "Grupo 12"
ANIO = 2025
PERIODO_INI = pd.Timestamp(f"{ANIO}-01-01")    # el caso analiza el año 2025 (README y API del IPC 2025)
PERIODO_FIN = pd.Timestamp(f"{ANIO}-12-31")
W_PRINCIPAL = 90                                # días: ventana de actividad elegida
W_ALTERNATIVA = 30                              # días: creencia verbalizada por la gerencia
VENTANAS_SENSIBILIDAD = [30, 45, 60, 75, 90]
FECHA_CIERRE = PERIODO_FIN
# Fecha base de la métrica: primer cierre de mes con W_PRINCIPAL días de historia dentro del período.
FECHA_BASE = (PERIODO_INI + pd.Timedelta(days=W_PRINCIPAL - 1)) + pd.offsets.MonthEnd(0)

# ---------------- Rutas ----------------
RAW, PROC, DOCS = "data/raw", "data/processed", "docs"
FIG, TAB, CAL = "outputs/figuras", "outputs/tablas", "outputs/calidad"
for carpeta in (PROC, FIG, TAB, CAL, DOCS):
    os.makedirs(carpeta, exist_ok=True)

FUENTE_PIE = f"Fuente: origen_pos.sqlite, boletas {ANIO} de clientes identificados. Elaboración propia, {GRUPO}."
print(f"RESULTADO: período {PERIODO_INI:%d-%m-%Y} a {PERIODO_FIN:%d-%m-%Y} · ventana principal {W_PRINCIPAL} días, "
      f"alternativa {W_ALTERNATIVA} días · fecha base de la métrica {FECHA_BASE:%d-%m-%Y}")

## 1. Extracción de las cuatro fuentes
Se leen las fuentes tal como vienen, sin transformarlas.

### 1.1 Base del punto de venta (SQLite)
**Objetivo:** leer todas las tablas de la base, sea cual sea su nombre, y mostrar su tamaño.

In [ ]:
con = sqlite3.connect(f"{RAW}/origen_pos.sqlite")
tablas_pos = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)["name"].tolist()
pos = {t: pd.read_sql(f'SELECT * FROM "{t}"', con) for t in tablas_pos}
con.close()
print("RESULTADO:")
print(pd.DataFrame([{"tabla": t, "filas": len(df), "columnas": ", ".join(df.columns)} for t, df in pos.items()]).to_string(index=False))

### 1.2 Maestro de productos (CSV)
**Objetivo:** leer el CSV detectando su codificación y separador en vez de suponerlos. Todo se lee como texto para que la sección 2 pueda revisar los formatos originales.

In [ ]:
def leer_csv_robusto(ruta):
    contenido = open(ruta, "rb").read()
    for codif in ("utf-8", "cp1252", "latin-1"):
        try:
            texto = contenido.decode(codif)
            break
        except UnicodeDecodeError:
            continue
    sep = csv.Sniffer().sniff(texto.splitlines()[0], delimiters=";,|\t").delimiter
    return pd.read_csv(io.StringIO(texto), sep=sep, dtype=str, keep_default_na=False), codif, sep

maestro_raw, codif_maestro, sep_maestro = leer_csv_robusto(f"{RAW}/origen_maestro_productos.csv")
print(f"RESULTADO: {len(maestro_raw):,} filas · codificación detectada {codif_maestro} · separador '{sep_maestro}'")
maestro_raw.head()

### 1.3 Promociones (JSON)
**Objetivo:** leer el archivo de campañas y aplanarlo en dos tablas: campañas y SKU por campaña.

In [ ]:
with open(f"{RAW}/origen_promociones.json", encoding="utf-8") as f:
    promos_raw = json.load(f)
campanas_raw = pd.DataFrame([{k: v for k, v in c.items() if k != "skus"} for c in promos_raw["campanas"]])
campana_sku_raw = pd.DataFrame([{"campana_id": c["campana_id"], "sku": s} for c in promos_raw["campanas"] for s in c["skus"]])
print(f"RESULTADO: {len(campanas_raw)} campañas, {len(campana_sku_raw)} pares campaña-SKU")
campanas_raw

### 1.4 IPC mensual desde la API de mindicador.cl, con respaldo real
**Objetivo:** obtener la serie del IPC y asegurar que el notebook corra aunque la API no responda.

Funciona así:
1. Se consulta la API. Si responde y la serie es válida (12 meses distintos con valores numéricos), **se guarda la respuesta original, sin modificar**, en `data/raw/ipc_<año>_respaldo_api.json`, junto con la fecha y hora de la consulta.
2. Si la API no responde, se usa ese respaldo. Si todavía no existe ningún respaldo de la API, se usa `ipc_<año>_respaldo_manual.json` (valores oficiales del INE copiados a mano en el formato de la API).
3. **Para que el respaldo quede en el repositorio**, hay que subir el archivo una vez. En Colab, poner `DESCARGAR_RESPALDO_IPC = True`: el navegador descarga el archivo y se sube a `data/raw/` en GitHub (*Add file → Upload files*). Colab solo sube el notebook cuando se guarda en GitHub; los demás archivos que crea la sesión se pierden al cerrarla.

In [ ]:
DESCARGAR_RESPALDO_IPC = False   # True: en Colab, descarga el respaldo para subirlo al repositorio

URL_IPC = f"https://mindicador.cl/api/ipc/{ANIO}"
RESPALDO_API = f"{RAW}/ipc_{ANIO}_respaldo_api.json"
RESPALDO_MANUAL = f"{RAW}/ipc_{ANIO}_respaldo_manual.json"

def validar_serie_ipc(data):
    meses = pd.to_datetime([s["fecha"] for s in data["serie"]], utc=True).tz_localize(None).to_period("M")
    meses = [m for m in meses if m.year == ANIO]
    valores_ok = all(isinstance(s["valor"], (int, float)) for s in data["serie"])
    if len(set(meses)) != 12 or not valores_ok:
        raise ValueError(f"serie inválida: {len(set(meses))} meses distintos de {ANIO}, valores numéricos={valores_ok}")

def obtener_ipc():
    try:
        r = requests.get(URL_IPC, timeout=20)
        r.raise_for_status()
        data = r.json()
        validar_serie_ipc(data)
        anterior = json.load(open(RESPALDO_API, encoding="utf-8"))["respuesta"] if os.path.exists(RESPALDO_API) else None
        cambio = anterior is None or anterior.get("serie") != data.get("serie")
        if cambio:
            with open(RESPALDO_API, "w", encoding="utf-8") as f:
                json.dump({"url": URL_IPC, "consultado_en": datetime.now().isoformat(timespec="seconds"),
                           "respuesta": data}, f, ensure_ascii=False, indent=1)
        return data, "API mindicador.cl", cambio
    except Exception as e:
        print(f"La API no respondió o devolvió datos inválidos: {e}")
        if os.path.exists(RESPALDO_API):
            resp = json.load(open(RESPALDO_API, encoding="utf-8"))
            print(f"Se usa el respaldo de la API consultado el {resp['consultado_en']}.")
            return resp["respuesta"], f"respaldo de la API ({resp['consultado_en'][:10]})", False
        print("No hay respaldo de la API; se usa el respaldo manual con valores del INE.")
        return json.load(open(RESPALDO_MANUAL, encoding="utf-8")), "respaldo manual (valores INE)", False

ipc_raw, FUENTE_IPC, respaldo_actualizado = obtener_ipc()
if respaldo_actualizado:
    print(f"Se guardó un respaldo nuevo de la API en {RESPALDO_API}. Súbanlo al repositorio para que quede disponible sin conexión.")
    if EN_COLAB and DESCARGAR_RESPALDO_IPC:
        from google.colab import files
        files.download(RESPALDO_API)
print(f"RESULTADO: IPC leído desde {FUENTE_IPC} · {len(ipc_raw['serie'])} registros · unidad: {ipc_raw.get('unidad_medida')}")

## 2. Detección de datos sucios
Esta sección **no modifica ningún dato**. Revisa cada fuente con chequeos genéricos y deja todo lo que encuentra en `outputs/calidad/`: un archivo por problema con las filas afectadas, y un resumen general al final. Las decisiones sobre qué hacer con cada problema se toman en la sección 3.

Primero se declaran las **reglas que deberían cumplir los datos** (claves, relaciones, formatos esperados). Los chequeos comparan los datos contra esas reglas, así que funcionan igual con otro año o con otra extracción.

In [ ]:
# Claves primarias esperadas: cada combinación debería ser única
CLAVES = {
    "boletas": ["boleta_id"], "boleta_detalle": ["boleta_id", "linea"], "clientes": ["cliente_id"],
    "tiendas": ["tienda_id"], "comunas": ["comuna_id"], "regiones": ["region_id"], "medios_pago": ["medio_id"],
    "maestro": ["sku"],
}
# Relaciones esperadas: (tabla, columna) debe existir en (tabla_referida, columna_referida)
RELACIONES = [
    ("comunas", "region_id", "regiones", "region_id"),
    ("tiendas", "comuna_id", "comunas", "comuna_id"),
    ("clientes", "comuna_id", "comunas", "comuna_id"),
    ("boletas", "tienda_id", "tiendas", "tienda_id"),
    ("boletas", "cliente_id", "clientes", "cliente_id"),
    ("boletas", "medio_id", "medios_pago", "medio_id"),
    ("boleta_detalle", "boleta_id", "boletas", "boleta_id"),
    ("boleta_detalle", "sku", "maestro", "sku"),
]
TABLAS = {**pos, "maestro": maestro_raw}
HALLAZGOS = []   # resumen de todos los chequeos

def hallazgo(chequeo, tabla, columna, afectados, total, detalle="", archivo=""):
    HALLAZGOS.append({"chequeo": chequeo, "tabla": tabla, "columna": columna, "afectados": int(afectados),
                      "total": int(total), "pct": round(100 * afectados / total, 2) if total else 0.0,
                      "detalle": detalle, "archivo": archivo})

def guardar_calidad(df, nombre):
    ruta = f"{CAL}/{nombre}.csv"
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    return ruta

print("RESULTADO: reglas declaradas →", len(CLAVES), "claves y", len(RELACIONES), "relaciones")

### 2.1 Perfil de columnas y formatos mixtos
**Objetivo:** encontrar columnas donde conviven varios formatos de escritura, sin saber de antemano cuáles son.

**Cómo funciona.** Cada valor se transforma en su *firma*: los dígitos se reemplazan por `9` y las letras por `a`/`A`. Por ejemplo, `2025-01-28` → `9999-99-99`, `28/01/2025` → `99/99/9999`, `$1.168` → `$9.999`. Luego se agrupan las firmas en *familias* (colapsando las rachas de dígitos: `9.999` y `99.999` son la misma familia `9.9`). Una columna mayoritariamente numérica o de fechas con más de una familia tiene formatos mixtos.

In [ ]:
def firma(v):
    s = re.sub(r"\d", "9", str(v))
    s = re.sub(r"[A-ZÁÉÍÓÚÑ]+", "A", s)
    return re.sub(r"[a-záéíóúñü]+", "a", s)

def familia(v):
    # el signo no es un formato distinto (los negativos se revisan en 2.8)
    return re.sub(r"9+", "9", re.sub(r"^\s*-", "", firma(v)))

perfil = []
for t, df in TABLAS.items():
    for col in df.columns:
        s = df[col]
        no_nulos = s[s.notna() & (s.astype(str).str.strip() != "")].astype(str)
        if no_nulos.empty:
            continue
        fams = no_nulos.map(familia).value_counts()
        es_numerico_o_fecha = (no_nulos.str.contains(r"\d") & (no_nulos.str.count(r"[A-Za-zÁÉÍÓÚáéíóúñÑ]") <= 3)).mean() >= 0.9
        perfil.append({"tabla": t, "columna": col, "filas": len(s), "nulos_o_vacios": len(s) - len(no_nulos),
                       "valores_distintos": no_nulos.nunique(), "n_familias_formato": len(fams),
                       "familias_principales": " | ".join(f"{k} ({v})" for k, v in fams.head(5).items()),
                       "formato_mixto": bool(es_numerico_o_fecha and len(fams) > 1)})
perfil = pd.DataFrame(perfil)
ruta = guardar_calidad(perfil, "01_perfil_columnas")
mixtos = perfil[perfil["formato_mixto"]]
for _, r in mixtos.iterrows():
    hallazgo("Formatos mixtos en la columna", r.tabla, r.columna, r.filas - r.nulos_o_vacios, r.filas, r.familias_principales, ruta)
print(f"RESULTADO: {len(perfil)} columnas perfiladas; {len(mixtos)} con formatos mixtos (detalle en {ruta}):")
mixtos[["tabla", "columna", "n_familias_formato", "familias_principales"]]

### 2.2 Filas que no corresponden al contenido de la tabla
**Objetivo:** detectar filas "basura" (encabezados repetidos, totales, notas) comparando cada clave con la firma dominante de su columna. Una clave que no se parece al resto de las claves no es un registro.

In [ ]:
def filas_fuera_de_patron(df, col, umbral=0.9):
    firmas = df[col].astype(str).str.strip().map(firma)
    dominante, share = firmas.value_counts(normalize=True).index[0], firmas.value_counts(normalize=True).iloc[0]
    return (firmas != dominante) if share >= umbral else pd.Series(False, index=df.index), dominante

basura_maestro, firma_sku = filas_fuera_de_patron(maestro_raw, "sku")
ruta = guardar_calidad(maestro_raw[basura_maestro], "02_maestro_filas_fuera_de_patron")
hallazgo("Fila cuya clave no sigue el patrón de la columna", "maestro", "sku", basura_maestro.sum(), len(maestro_raw),
         f"patrón dominante '{firma_sku}'", ruta)
print(f"RESULTADO: patrón dominante del SKU = '{firma_sku}'; {basura_maestro.sum()} filas no lo cumplen:")
maestro_raw[basura_maestro]

### 2.3 Duplicados y claves repetidas
**Objetivo:** distinguir dos problemas distintos: filas idénticas repetidas (la misma transacción cargada dos veces) y claves repetidas con contenido distinto (dos registros que dicen ser el mismo pero no coinciden).

In [ ]:
resumen_dup = []
for t, clave in CLAVES.items():
    df = TABLAS[t] if t != "maestro" else maestro_raw[~basura_maestro]
    df_cmp = df.apply(lambda s: s.astype(str).str.strip().str.upper()) if t == "maestro" else df
    exactos = df_cmp.duplicated(keep="first")
    sin_exactos = df_cmp[~exactos]
    clave_rep = sin_exactos.duplicated(clave, keep=False)
    if exactos.any():
        ruta = guardar_calidad(df[df_cmp.duplicated(keep=False)].sort_values(clave), f"03_{t}_duplicados_exactos")
        hallazgo("Filas duplicadas exactas", t, "+".join(clave), exactos.sum(), len(df), "", ruta)
    if clave_rep.any():
        ruta = guardar_calidad(df.loc[sin_exactos[clave_rep].index].sort_values(clave), f"03_{t}_clave_repetida")
        hallazgo("Clave repetida con contenido distinto", t, "+".join(clave), clave_rep.sum(), len(df), "", ruta)
    resumen_dup.append({"tabla": t, "clave": "+".join(clave), "filas": len(df),
                        "duplicados_exactos": int(exactos.sum()), "filas_con_clave_repetida": int(clave_rep.sum())})
print("RESULTADO:")
pd.DataFrame(resumen_dup)

**Lectura.** Si una tabla tiene claves repetidas con contenido distinto, conviene abrir su archivo en `outputs/calidad/` para ver si son registros diferentes mal numerados o versiones distintas del mismo registro.

### 2.4 Integridad referencial
**Objetivo:** encontrar valores que apuntan a registros que no existen (por ejemplo, una comuna con una región inexistente). Los nulos se cuentan aparte, porque no son un error de referencia (una boleta sin cliente es una compra anónima).

In [ ]:
def norm_clave(s):
    return s.astype(str).str.strip().str.upper().str.replace(r"\.0$", "", regex=True)

maestro_sin_basura = maestro_raw[~basura_maestro]
resumen_fk = []
for t, col, tref, colref in RELACIONES:
    df = TABLAS[t]
    ref = maestro_sin_basura if tref == "maestro" else TABLAS[tref]
    nulos = df[col].isna() | (df[col].astype(str).str.strip() == "")
    huerf = ~nulos & ~norm_clave(df[col]).isin(set(norm_clave(ref[colref])))
    if huerf.any():
        ruta = guardar_calidad(df[huerf], f"04_{t}_{col}_huerfanos")
        valores = ", ".join(map(str, sorted(df.loc[huerf, col].astype(str).unique())[:10]))
        hallazgo("Referencia huérfana", t, col, huerf.sum(), len(df), f"no existe en {tref}.{colref}: {valores}", ruta)
    resumen_fk.append({"relación": f"{t}.{col} → {tref}.{colref}", "filas": len(df), "nulos": int(nulos.sum()),
                       "huérfanos": int(huerf.sum()),
                       "valores_huérfanos": ", ".join(map(str, sorted(df.loc[huerf, col].astype(str).unique())[:10]))})
print("RESULTADO:")
pd.DataFrame(resumen_fk)

### 2.5 Fechas: formatos, orden día-mes y rango
**Objetivo:** convertir las columnas de fecha reconociendo todos los formatos presentes y **decidir con los datos** si los formatos ambiguos (`99/99/9999`, `99-99-99`) son día-mes o mes-día: si el primer número supera 12 en algunos casos y el segundo nunca, el orden es día-mes. Luego se revisa el rango de fechas contra el período del análisis.

In [ ]:
MESES_TEXTO = {"ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6, "jul": 7, "ago": 8,
               "sep": 9, "set": 9, "oct": 10, "nov": 11, "dic": 12}

def evidencia_orden(serie, patron):
    m = serie.astype(str).str.strip().str.extract(patron).dropna().astype(int)
    return int((m[0] > 12).sum()), int((m[1] > 12).sum())

def parsear_fechas(serie, nombre):
    s = serie.astype(str).str.strip()
    evid = {}
    for etiqueta, patron in [("99/99/9999", r"^(\d{1,2})/(\d{1,2})/\d{4}$"), ("99-99-99", r"^(\d{1,2})-(\d{1,2})-\d{2}$")]:
        p1, p2 = evidencia_orden(s, patron)
        orden = "día-mes" if p1 > 0 and p2 == 0 else ("mes-día" if p2 > 0 and p1 == 0 else "indeterminado")
        evid[etiqueta] = (p1, p2, orden)
        print(f"  {nombre} · formato {etiqueta}: primer número > 12 en {p1} casos, segundo > 12 en {p2} → orden {orden}")
    dia_primero = all(o != "mes-día" for _, _, o in evid.values())

    def uno(x):
        try:
            if re.fullmatch(r"\d{4}-\d{1,2}-\d{1,2}", x):
                return pd.Timestamp(x)
            m = re.fullmatch(r"(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})", x)
            if m:
                a, b, y = int(m[1]), int(m[2]), int(m[3])
                y = y + 2000 if y < 100 else y
                return pd.Timestamp(y, b, a) if dia_primero else pd.Timestamp(y, a, b)
            m = re.fullmatch(r"(\d{1,2})[-/ ]([A-Za-z]{3})[a-z]*[-/ ](\d{2,4})", x)
            if m and m[2].lower() in MESES_TEXTO:
                y = int(m[3]); y = y + 2000 if y < 100 else y
                return pd.Timestamp(y, MESES_TEXTO[m[2].lower()], int(m[1]))
        except ValueError:
            return pd.NaT
        return pd.NaT
    return s.map(uno)

print("Evidencia del orden día-mes:")
fecha_boleta = parsear_fechas(pos["boletas"]["fecha"], "boletas.fecha")
fecha_alta = parsear_fechas(pos["clientes"]["fecha_alta"], "clientes.fecha_alta")

for nombre, serie, tabla in [("fecha", fecha_boleta, "boletas"), ("fecha_alta", fecha_alta, "clientes")]:
    no_parse = serie.isna()
    if no_parse.any():
        ruta = guardar_calidad(pos[tabla][no_parse], f"05_{tabla}_{nombre}_no_interpretable")
        hallazgo("Fecha no interpretable", tabla, nombre, no_parse.sum(), len(serie), "", ruta)
    print(f"  {tabla}.{nombre}: rango {serie.min():%d-%m-%Y} a {serie.max():%d-%m-%Y} · no interpretables: {no_parse.sum()}")

fuera = (fecha_boleta < PERIODO_INI) | (fecha_boleta > PERIODO_FIN)
ruta = guardar_calidad(pos["boletas"][fuera].assign(fecha_interpretada=fecha_boleta[fuera]), "05_boletas_fuera_de_periodo")
hallazgo("Fecha fuera del período de análisis", "boletas", "fecha", fuera.sum(), len(fuera),
         f"período {PERIODO_INI:%d-%m-%Y} a {PERIODO_FIN:%d-%m-%Y}", ruta)

boletas_por_mes = fecha_boleta.dt.to_period("M").value_counts().sort_index().rename("boletas").to_frame()
boletas_por_mes["dentro_del_periodo"] = [(p.start_time >= PERIODO_INI) and (p.end_time <= PERIODO_FIN + pd.Timedelta(days=1)) for p in boletas_por_mes.index]
guardar_calidad(boletas_por_mes.reset_index(names="mes"), "05_boletas_por_mes")
dentro, fuera_m = boletas_por_mes[boletas_por_mes.dentro_del_periodo], boletas_por_mes[~boletas_por_mes.dentro_del_periodo]
print(f"\nRESULTADO: {fuera.sum()} boletas fuera del período ({100*fuera.mean():.1f}%). "
      f"Boletas por mes dentro del período: promedio {dentro['boletas'].mean():.0f}; "
      f"fuera del período: {len(fuera_m)} meses con promedio {fuera_m['boletas'].mean() if len(fuera_m) else 0:.0f}.")
boletas_por_mes.T

**Lectura.** Si los meses fuera del período tienen un volumen comparable a los de dentro, la extracción cubre más tiempo y conviene ampliar el período. Si tienen muy pocas boletas, son registros sueltos (probablemente errores de fecha) y no una extensión real de los datos.

### 2.6 Números escritos como texto
**Objetivo:** convertir montos, precios y descuentos que vienen como texto en distintos formatos, decidiendo **con los datos** qué significa el punto. Si en todos los casos el punto va seguido de exactamente tres dígitos, es separador de miles; si no, es decimal.

In [ ]:
def evidencia_punto(serie):
    grupos = serie.astype(str).str.findall(r"\.(\d+)").explode().dropna()
    return (grupos.str.len() == 3).mean() if len(grupos) else np.nan

def parsear_numero(serie, punto_es_miles):
    s = serie.astype(str).str.strip().str.replace(" ", "").str.replace("$", "", regex=False)
    if punto_es_miles:
        s = s.str.replace(r"^(-?)\.(\d{3})$", r"\1\2", regex=True)   # '-.769' → '-769' (grupo de miles sin dígito delante)
        s = s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    else:
        s = s.str.replace(",", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

def parsear_descuento(serie):
    s = serie.astype(str).str.strip()
    pct = s.str.endswith("%")
    v = pd.to_numeric(s.str.rstrip("%").str.replace(",", ".", regex=False), errors="coerce")
    # '10%' o '10' → 0,10 ; '0,10' → 0,10
    return np.where(pct | (v >= 1), v / 100, v)

columnas_numericas = [("boleta_detalle", "precio_unitario"), ("boleta_detalle", "monto_linea"),
                      ("maestro", "costo_unitario"), ("maestro", "precio_lista")]
NUM = {}
for t, col in columnas_numericas:
    df = TABLAS[t] if t != "maestro" else maestro_sin_basura
    share3 = evidencia_punto(df[col])
    punto_es_miles = bool(share3 == 1.0) if not np.isnan(share3) else True
    NUM[(t, col)] = parsear_numero(df[col], punto_es_miles)
    fallidos = NUM[(t, col)].isna() & df[col].astype(str).str.strip().ne("")
    especiales = df[col].astype(str).str.fullmatch(r"-\.\d{3}")
    print(f"  {t}.{col}: punto seguido de 3 dígitos en {100*share3:.0f}% de los casos → punto = "
          f"{'miles' if punto_es_miles else 'decimal'} · no convertibles: {fallidos.sum()} · formato '-.999': {especiales.sum()}")
    if fallidos.any():
        ruta = guardar_calidad(df[fallidos], f"06_{t}_{col}_no_convertible")
        hallazgo("Número no convertible", t, col, fallidos.sum(), len(df), "", ruta)
    if especiales.any():
        ruta = guardar_calidad(df[especiales], f"06_{t}_{col}_negativo_sin_entero")
        hallazgo("Negativo escrito sin dígito antes del punto ('-.999')", t, col, especiales.sum(), len(df),
                 "se interpreta como -999 (grupo de miles incompleto)", ruta)

det_raw = pos["boleta_detalle"]
DESC = pd.Series(parsear_descuento(det_raw["descuento"]), index=det_raw.index)
desc_notaciones = det_raw["descuento"].astype(str).map(familia).value_counts()
print("  boleta_detalle.descuento: notaciones encontradas →", desc_notaciones.to_dict(),
      "| valores resultantes:", sorted(pd.Series(DESC).round(4).unique().tolist()))
print("RESULTADO: columnas numéricas convertidas; la regla de negocio de 2.7 valida que la interpretación sea correcta.")

### 2.7 Reglas de negocio del detalle de boletas
**Objetivo:** verificar que cada línea cumpla `monto_linea = cantidad × precio_unitario × (1 − descuento)`, y clasificar las diferencias:
- **Exacta:** coincide.
- **Truncamiento:** la diferencia es menor a $1 y el monto es el cálculo sin decimales. Pasa cuando el descuento genera centavos.
- **Discrepancia:** diferencia de $1 o más.

Para las discrepancias, se busca **cuál de los dos campos está mal comparándolos con una fuente independiente**: el precio de lista del maestro de productos. El campo que se aleja del precio de lista en las discrepancias, pero no en las líneas exactas, es el sospechoso.

También se revisa si el redondeo de pagos en efectivo (ley chilena de redondeo, que ajusta el total a la decena) explica las diferencias: si así fuera, las diferencias serían de $5 o menos y se concentrarían en boletas pagadas en efectivo.

In [ ]:
det_chk = det_raw.assign(
    precio=NUM[("boleta_detalle", "precio_unitario")], monto=NUM[("boleta_detalle", "monto_linea")], desc=DESC,
    sku_n=det_raw["sku"].astype(str).str.strip().str.upper())
det_chk["calculado"] = det_chk["cantidad"] * det_chk["precio"] * (1 - det_chk["desc"])
det_chk["diferencia"] = det_chk["monto"] - det_chk["calculado"]
det_chk["tipo"] = np.select(
    [det_chk["diferencia"] == 0, (det_chk["diferencia"].abs() < 1) & (det_chk["monto"] == np.trunc(det_chk["calculado"]))],
    ["exacta", "truncamiento"], "discrepancia")
conteo_tipo = det_chk["tipo"].value_counts()

precio_lista = pd.Series(NUM[("maestro", "precio_lista")].values, index=maestro_sin_basura["sku"].str.strip().str.upper())
precio_lista = precio_lista[~precio_lista.index.duplicated()]
det_chk["precio_lista"] = det_chk["sku_n"].map(precio_lista)
det_chk["unitario_implicito"] = det_chk["monto"] / (det_chk["cantidad"] * (1 - det_chk["desc"]))
comparacion = det_chk.dropna(subset=["precio_lista"]).groupby(det_chk["tipo"] == "discrepancia").apply(lambda g: pd.Series({
    "precio_unitario / precio_lista (mediana)": (g["precio"] / g["precio_lista"]).median(),
    "monto_linea implícito / precio_lista (mediana)": (g["unitario_implicito"] / g["precio_lista"]).median()}))
comparacion.index = comparacion.index.map({False: "líneas sin discrepancia", True: "líneas con discrepancia"})

boletas_medio = pos["boletas"].drop_duplicates("boleta_id").set_index("boleta_id")["medio_id"]
medios = pos["medios_pago"].set_index("medio_id")["medio"]
det_chk["medio"] = det_chk["boleta_id"].map(boletas_medio).map(medios)
tasa_por_medio = det_chk.groupby("medio")["tipo"].apply(lambda s: round(100 * (s == "discrepancia").mean(), 1))
dif_disc = det_chk.loc[det_chk["tipo"] == "discrepancia", "diferencia"].abs()

if (det_chk["tipo"] == "discrepancia").any():
    ruta = guardar_calidad(det_chk[det_chk["tipo"] == "discrepancia"], "07_detalle_monto_discrepante")
    hallazgo("monto_linea no cumple cantidad × precio × (1 − descuento)", "boleta_detalle", "monto_linea",
             conteo_tipo.get("discrepancia", 0), len(det_chk), "diferencia de $1 o más", ruta)

print("RESULTADO:")
print(" Tipo de línea:", conteo_tipo.to_dict())
print(f" Discrepancias: diferencia absoluta mediana ${dif_disc.median():,.0f}, mínima ${dif_disc.min():,.0f}" if len(dif_disc) else " Sin discrepancias")
print(" % de líneas con discrepancia por medio de pago:", tasa_por_medio.to_dict())
print(" Comparación con el precio de lista del maestro (fuente independiente):")
comparacion.round(3)

**Lectura.** Si en las líneas con discrepancia `precio_unitario` sigue igual de cerca del precio de lista que en las líneas sanas, pero el precio implícito en `monto_linea` se aleja, el campo dañado es `monto_linea`. Si las diferencias fueran por el redondeo en efectivo, serían de $5 o menos y la tasa sería mayor en "Efectivo".

### 2.8 Cantidades negativas: ¿devoluciones o signo invertido?
**Objetivo:** decidir con evidencia qué representan las líneas con cantidad negativa. Si fueran devoluciones, se esperaría que (a) el cliente hubiera comprado ese mismo producto antes, más seguido que en una línea cualquiera, o (b) que el mismo producto apareciera en positivo en la misma boleta (anulación de un ítem). Si no se cumple nada de eso y la distribución de cantidades es igual a la de las líneas positivas, lo más probable es un error de signo.

In [ ]:
neg = det_chk["cantidad"] < 0
signo_coherente = (np.sign(det_chk["cantidad"]) == np.sign(det_chk["monto"])) | (det_chk["monto"] == 0)
fechas_b = pd.Series(fecha_boleta.values, index=pos["boletas"]["boleta_id"]).groupby(level=0).first()
cliente_b = pos["boletas"].drop_duplicates("boleta_id").set_index("boleta_id")["cliente_id"]
lin = det_chk.assign(fecha=det_chk["boleta_id"].map(fechas_b), cliente=det_chk["boleta_id"].map(cliente_b))
lin_ident = lin.dropna(subset=["cliente"])
compras_pos = lin_ident[lin_ident["cantidad"] > 0][["cliente", "sku_n", "fecha"]]
primera_compra_sku = compras_pos.groupby(["cliente", "sku_n"])["fecha"].min()

def compro_antes(df):
    claves = list(zip(df["cliente"], df["sku_n"]))
    prim = primera_compra_sku.reindex(claves).values
    return (pd.Series(prim, index=df.index) < df["fecha"]).mean()

pct_previo_neg = compro_antes(lin_ident[lin_ident["cantidad"] < 0]) if (lin_ident["cantidad"] < 0).any() else np.nan
pct_previo_base = compro_antes(lin_ident[lin_ident["cantidad"] > 0].sample(min(2000, (lin_ident["cantidad"] > 0).sum()), random_state=1))
pos_mismo = det_chk[det_chk["cantidad"] > 0].groupby("boleta_id")["sku_n"].apply(set)
pct_mismo_sku = det_chk[neg].apply(lambda r: r["sku_n"] in pos_mismo.get(r["boleta_id"], set()), axis=1).mean() if neg.any() else np.nan
dist = pd.DataFrame({"líneas positivas": det_chk.loc[det_chk["cantidad"] > 0, "cantidad"].clip(upper=6).value_counts(normalize=True),
                     "líneas negativas (valor absoluto)": det_chk.loc[neg, "cantidad"].abs().clip(upper=6).value_counts(normalize=True)}).fillna(0).round(3)
por_boleta = det_chk.groupby("boleta_id")["cantidad"].agg(n="size", neg=lambda s: (s < 0).sum())

EVIDENCIA_DEVOLUCION = (pct_previo_neg > pct_previo_base + 0.10) or (pct_mismo_sku > 0.5)
if neg.any():
    ruta = guardar_calidad(det_chk[neg], "08_detalle_cantidad_negativa")
    hallazgo("Cantidad negativa", "boleta_detalle", "cantidad", neg.sum(), len(det_chk),
             f"devolución probable: {EVIDENCIA_DEVOLUCION}", ruta)
print("RESULTADO:")
print(f" Líneas negativas: {neg.sum()} · signo de cantidad y monto coherente: {100*signo_coherente[neg].mean():.0f}%")
print(f" Boletas con líneas negativas: {(por_boleta.neg > 0).sum()} (solo negativas: {((por_boleta.neg > 0) & (por_boleta.neg == por_boleta.n)).sum()}, mixtas: {((por_boleta.neg > 0) & (por_boleta.neg < por_boleta.n)).sum()})")
print(f" (a) El cliente compró ese SKU antes: {100*pct_previo_neg:.0f}% en líneas negativas vs {100*pct_previo_base:.0f}% en líneas positivas al azar")
print(f" (b) El mismo SKU aparece en positivo en la misma boleta: {100*pct_mismo_sku:.0f}%")
print(f" ¿Hay evidencia de devolución? {EVIDENCIA_DEVOLUCION}")
dist.T

### 2.9 Columnas redundantes que se contradicen
**Objetivo:** las boletas traen columnas que se pueden derivar de otras (`mes` sale de `fecha`; `region` sale de la tienda). Se guardan las filas donde no coinciden y se busca **cuál de las dos versiones es la confiable**:
- Para `mes` vs `fecha`: las líneas con el descuento de una campaña deberían caer dentro de las fechas de esa campaña. Se mide qué porcentaje cae dentro según `fecha` y según `mes`.
- Para `region`: se compara con la región de la tienda y con la región del cliente, para ver si la columna representa otra cosa.

In [ ]:
b = pos["boletas"].copy()
b["fecha_interpretada"] = fecha_boleta
b["mes_desde_fecha"] = fecha_boleta.dt.month
mes_inc = pd.to_numeric(b["mes"], errors="coerce") != b["mes_desde_fecha"]
ruta_mes = guardar_calidad(b[mes_inc][["boleta_id", "fecha", "fecha_interpretada", "mes", "mes_desde_fecha", "tienda_id", "canal"]],
                           "09_boletas_mes_inconsistente")
hallazgo("Columna 'mes' distinta del mes de la fecha", "boletas", "mes", mes_inc.sum(), len(b), "", ruta_mes)

# ¿Cuál es confiable? Prueba con las campañas
camp_val = []
for _, c in campanas_raw.iterrows():
    d_c = float(str(c["descuento_pct"]).replace("%", "").replace(",", ".")); d_c = d_c / 100 if d_c >= 1 else d_c
    skus_c = set(campana_sku_raw.loc[campana_sku_raw["campana_id"] == c["campana_id"], "sku"].str.strip().str.upper())
    ini, fin = pd.Timestamp(c["fecha_ini"]), pd.Timestamp(c["fecha_fin"])
    l = det_chk[det_chk["sku_n"].isin(skus_c) & (det_chk["desc"].round(4) == round(d_c, 4))].merge(
        b[["boleta_id", "fecha_interpretada", "mes"]].drop_duplicates("boleta_id"), on="boleta_id")
    meses_camp = set(pd.period_range(ini, fin, freq="M").month)
    camp_val.append({"campaña": c["campana_id"], "descuento": d_c, "líneas con ese descuento": len(l),
                     "% dentro según fecha": round(100 * l["fecha_interpretada"].between(ini.normalize(), fin).mean(), 1) if len(l) else np.nan,
                     "% dentro según columna mes": round(100 * pd.to_numeric(l["mes"]).isin(meses_camp).mean(), 1) if len(l) else np.nan})
camp_val = pd.DataFrame(camp_val)

# Región
geo = pos["tiendas"].merge(pos["comunas"], on="comuna_id", how="left").merge(pos["regiones"], on="region_id", how="left")
reg_tienda = b["tienda_id"].map(geo.set_index("tienda_id")["region"])
geo_cli = pos["clientes"].merge(pos["comunas"], on="comuna_id", how="left").merge(pos["regiones"], on="region_id", how="left")
reg_cliente = b["cliente_id"].map(geo_cli.set_index("cliente_id")["region"])
reg_inc = b["region"] != reg_tienda
b_reg = b.assign(region_tienda=reg_tienda, region_cliente=reg_cliente,
                 motivo=np.where(reg_tienda.isna(), "la tienda no tiene región válida (referencia huérfana)", "región distinta a la de la tienda"))
ruta_reg = guardar_calidad(b_reg[reg_inc][["boleta_id", "fecha", "tienda_id", "canal", "cliente_id", "region", "region_tienda",
                                           "region_cliente", "motivo"]], "09_boletas_region_inconsistente")
hallazgo("Columna 'region' distinta de la región de la tienda", "boletas", "region", reg_inc.sum(), len(b), "", ruta_reg)
solo_distinta = reg_inc & reg_tienda.notna()
coincide_cli = (b.loc[solo_distinta, "region"] == reg_cliente[solo_distinta]).mean()
base_cli = (b.loc[~reg_inc & reg_cliente.notna(), "region"] == reg_cliente[~reg_inc & reg_cliente.notna()]).mean()

print("RESULTADO:")
print(f" 'mes' distinto del mes de la fecha: {mes_inc.sum()} boletas ({100*mes_inc.mean():.1f}%) → {ruta_mes}")
print(f" 'region' distinta de la región de la tienda: {reg_inc.sum()} boletas → {ruta_reg}")
print(f"   de ellas, por tienda sin región válida: {reg_tienda[reg_inc].isna().sum()}; con región distinta: {solo_distinta.sum()}")
print(f"   en las de región distinta, la región coincide con la del cliente en {100*coincide_cli:.0f}% "
      f"(en boletas sin problema coincide en {100*base_cli:.0f}%: no representa la región del cliente)")
print(" Prueba de confiabilidad de 'fecha' vs 'mes' con las campañas:")
camp_val

**Lectura.** Si las líneas con descuento de campaña caen casi siempre dentro de la campaña según `fecha`, pero no según `mes`, la columna confiable es `fecha`. Una campaña sin líneas con su descuento indica que esa promoción no se registra como descuento en el POS (por ejemplo, un 2x1 puede registrarse como unidades gratis y no como porcentaje).

### 2.10 Clientes: RUT y fecha de alta
**Objetivo:** validar el RUT con su dígito verificador (algoritmo módulo 11) para detectar RUT mal escritos o sin dígito verificador, y comparar la fecha de alta con la primera compra registrada. Un cliente no debería tener compras identificadas antes de inscribirse.

In [ ]:
def dv_rut(cuerpo):
    s, m = 0, 2
    for d in reversed(str(cuerpo)):
        s += int(d) * m
        m = 2 if m == 7 else m + 1
    r = 11 - s % 11
    return "0" if r == 11 else "K" if r == 10 else str(r)

cli = pos["clientes"].copy()
rut = cli["rut"].astype(str).str.upper().str.replace(r"[\s.]", "", regex=True)   # quita espacios y puntos
con_guion = rut.str.contains("-")
cuerpo = np.where(con_guion, rut.str.split("-").str[0], rut.str[:-1])
dv = np.where(con_guion, rut.str.split("-").str[1], rut.str[-1])
cli["rut_valido"] = [c.isdigit() and dv_rut(c) == d for c, d in zip(cuerpo, dv)]
cli["rut_cuerpo"] = [c if v else (r if not g else c) for c, r, g, v in zip(cuerpo, rut, con_guion, cli["rut_valido"])]
sin_guion_valido = (~con_guion & cli["rut_valido"]).sum()
sin_guion_invalido = (~con_guion & ~cli["rut_valido"]).sum()
con_guion_invalido = (con_guion & ~cli["rut_valido"]).sum()
rut_dup = cli["rut_cuerpo"].duplicated(keep=False)
if (~cli["rut_valido"]).any():
    ruta = guardar_calidad(cli[~cli["rut_valido"]], "10_clientes_rut_invalido")
    hallazgo("RUT con dígito verificador inválido o ausente", "clientes", "rut", (~cli["rut_valido"]).sum(), len(cli),
             f"sin guion y válidos: {sin_guion_valido}; sin guion e inválidos: {sin_guion_invalido}; con guion e inválidos: {con_guion_invalido}", ruta)

fa = pd.Series(fecha_alta.values, index=cli["cliente_id"])
validas_tmp = pd.DataFrame({"cliente_id": pos["boletas"]["cliente_id"], "fecha": fecha_boleta}).dropna()
validas_tmp = validas_tmp[validas_tmp["fecha"].between(PERIODO_INI, PERIODO_FIN)]
primera = validas_tmp.groupby("cliente_id")["fecha"].min()
comp = pd.DataFrame({"fecha_alta": fa, "primera_compra": primera.reindex(fa.index)})
comp["dias_alta_a_primera_compra"] = (comp["primera_compra"] - comp["fecha_alta"]).dt.days
antes = comp["dias_alta_a_primera_compra"] < 0
ruta = guardar_calidad(comp[antes].reset_index(), "10_clientes_compra_antes_del_alta")
hallazgo("Compras identificadas anteriores a la fecha de alta", "clientes", "fecha_alta", antes.sum(), len(comp), "", ruta)
print("RESULTADO:")
print(f" RUT: con guion {con_guion.sum()}, sin guion {(~con_guion).sum()} · dígito verificador inválido o ausente: {(~cli['rut_valido']).sum()}"
      f" (sin guion pero con verificador correcto al final: {sin_guion_valido}) · RUT repetidos: {rut_dup.sum()}")
print(f" Fecha de alta: igual a la primera compra en {(comp['dias_alta_a_primera_compra'] == 0).sum()} clientes, "
      f"anterior en {(comp['dias_alta_a_primera_compra'] > 0).sum()}, posterior (compró antes de inscribirse) en {antes.sum()}; "
      f"altas anteriores al período: {(comp['fecha_alta'] < PERIODO_INI).sum()}")

### 2.11 Variantes de escritura en columnas de texto
**Objetivo:** encontrar categorías escritas de distintas formas (mayúsculas, tildes, abreviaciones) y proponer la forma canónica de manera automática:
1. Se normaliza el texto (minúsculas, sin tildes ni espacios sobrantes). Las variantes que quedan iguales son la misma categoría.
2. Las abreviaciones se emparejan con una forma completa si sus letras aparecen en orden dentro de ella (`LACT` → lácteos, `SNK` → snacks).
3. Lo que siga sin resolverse se asigna a la categoría que tiene su **subcategoría** en el resto de las filas.
4. Lo que no se pueda resolver así queda marcado para decisión manual.

In [ ]:
def normalizar_texto(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower().strip()

def es_subsecuencia(corta, larga):
    it = iter(larga)
    return corta[0] == larga[0] and all(ch in it for ch in corta)

def proponer_canonicos(df, col, col_contexto=None):
    v = df[col].astype(str).str.strip()
    norm = v.map(normalizar_texto)
    # Forma completa: tiene al menos una escritura que no está toda en mayúsculas (los códigos sí lo están)
    # y no es una abreviación de otra forma completa. La forma visible es la variante con tildes y mayúscula inicial.
    candidatas = [n for n in norm.unique() if (v[norm == n] != v[norm == n].str.upper()).any()]
    completas = [n for n in candidatas if not any(o != n and len(o) > len(n) and es_subsecuencia(n, o) for o in candidatas)]
    visible = {n: v[norm == n].map(lambda x: (sum(c in "áéíóúÁÉÍÓÚ" for c in x), x[0].isupper(), x)).max()[2] for n in completas}
    mapa, metodo = {}, {}
    for n in norm.unique():
        if n in completas:
            mapa[n], metodo[n] = visible[n], "misma palabra normalizada"
        else:
            cand = [c for c in completas if es_subsecuencia(n, c)]
            if len(cand) == 1:
                mapa[n], metodo[n] = visible[cand[0]], "abreviación"
    if col_contexto:
        ctx = df[col_contexto].astype(str).str.strip()
        resueltas = norm.map(mapa)
        mayoria = resueltas.groupby(ctx).agg(lambda s: s.dropna().mode().iloc[0] if s.notna().any() else None)
        for n in norm.unique():
            if n not in mapa:
                cats = ctx[norm == n].map(mayoria).dropna().unique()
                if len(cats) == 1:
                    mapa[n], metodo[n] = cats[0], f"categoría mayoritaria de su {col_contexto}"
    tabla = (pd.DataFrame({"original": v, "normalizado": norm}).value_counts().reset_index(name="filas")
             .assign(propuesta=lambda d: d["normalizado"].map(mapa), metodo=lambda d: d["normalizado"].map(metodo)))
    return tabla.sort_values(["propuesta", "filas"], ascending=[True, False])

variantes_cat = proponer_canonicos(maestro_sin_basura, "categoria", "subcategoria")
ruta = guardar_calidad(variantes_cat, "11_maestro_categoria_variantes")
no_canon = variantes_cat[variantes_cat["original"] != variantes_cat["propuesta"]]
hallazgo("Categoría escrita de forma no canónica", "maestro", "categoria", no_canon["filas"].sum(), len(maestro_sin_basura),
         f"{variantes_cat['original'].nunique()} escrituras para {variantes_cat['propuesta'].nunique()} categorías", ruta)
sin_resolver = variantes_cat[variantes_cat["propuesta"].isna()]

otras = []
for col in ["subcategoria", "marca", "proveedor"]:
    t = proponer_canonicos(maestro_sin_basura, col)
    n_var = t.groupby("normalizado")["original"].nunique()
    otras.append({"columna": col, "escrituras": t["original"].nunique(), "categorías normalizadas": len(n_var),
                  "con más de una escritura": int((n_var > 1).sum())})

activo_vals = maestro_sin_basura["activo"].astype(str).map(normalizar_texto).value_counts()
print("RESULTADO:")
print(f" categoria: {variantes_cat['original'].nunique()} escrituras distintas → {variantes_cat['propuesta'].nunique()} categorías propuestas; "
      f"sin resolver automáticamente: {len(sin_resolver)}")
print(" Otras columnas de texto:", otras)
print(" Valores de 'activo':", activo_vals.to_dict())
variantes_cat

### 2.12 Promociones y serie del IPC
**Objetivo:** revisar que los descuentos de las campañas tengan una sola notación, que sus SKU existan en el maestro, y que la serie del IPC tenga los 12 meses.

In [ ]:
desc_camp_notacion = campanas_raw["descuento_pct"].map(lambda x: type(x).__name__ + ":" + familia(x))
skus_maestro = set(maestro_sin_basura["sku"].str.strip().str.upper())
camp_sku_fuera = ~campana_sku_raw["sku"].str.strip().str.upper().isin(skus_maestro)
if desc_camp_notacion.nunique() > 1:
    hallazgo("Descuento de campaña en notaciones distintas", "promociones", "descuento_pct", len(campanas_raw), len(campanas_raw),
             ", ".join(desc_camp_notacion.unique()))
if camp_sku_fuera.any():
    ruta = guardar_calidad(campana_sku_raw[camp_sku_fuera], "12_promociones_sku_inexistente")
    hallazgo("SKU en promoción que no existe en el maestro", "promociones", "sku", camp_sku_fuera.sum(), len(campana_sku_raw), "", ruta)

ipc = pd.DataFrame(ipc_raw["serie"])
ipc["mes"] = pd.to_datetime(ipc["fecha"], utc=True).dt.tz_localize(None).dt.to_period("M")
ipc = ipc[ipc["mes"].dt.year == ANIO].sort_values("mes").reset_index(drop=True)
print("RESULTADO:")
print(f" Notaciones del descuento en campañas: {desc_camp_notacion.tolist()}")
print(f" SKU de campañas que no existen en el maestro: {campana_sku_raw.loc[camp_sku_fuera, 'sku'].tolist()}")
print(f" IPC: {ipc['mes'].nunique()} meses distintos de {ANIO}, de {ipc['mes'].min()} a {ipc['mes'].max()}")

### 2.13 Resumen de calidad
**Objetivo:** reunir en una tabla todos los problemas detectados. Este archivo es el punto de partida de la bitácora de supuestos del informe.

In [ ]:
resumen_calidad = pd.DataFrame(HALLAZGOS)
resumen_calidad.to_csv(f"{CAL}/00_resumen_calidad.csv", index=False, encoding="utf-8-sig")
print(f"RESULTADO: {len(resumen_calidad)} problemas detectados; archivos con las filas afectadas en {CAL}/")
resumen_calidad[["chequeo", "tabla", "columna", "afectados", "total", "pct", "detalle"]]

## 3. Decisiones de limpieza
Cada problema de la sección 2 recibe aquí una decisión. Las que se pueden tomar con evidencia de los datos se deciden de forma automática (`"auto"`) y se imprime el criterio; el grupo puede cambiarlas por otra opción. Las que necesitan conocimiento externo (por ejemplo, a qué región pertenece una comuna) se declaran a mano con su justificación.

**Objetivo de este bloque:** fijar todas las decisiones en un solo lugar.

In [ ]:
DECISIONES = {
    # 2.3 Duplicados exactos: se elimina la copia
    "duplicados_exactos": "eliminar",
    # 2.5 Fechas fuera del período: se excluyen del análisis (quedan guardadas en outputs/calidad/)
    "fuera_de_periodo": "excluir",
    # 2.7 Monto en discrepancia: "auto" elige el campo que coincide con el precio de lista del maestro
    #     opciones: "auto" | "recalcular" (cantidad × precio × (1 − desc), sin decimales) | "monto_linea"
    "monto_discrepante": "auto",
    # 2.8 Cantidades negativas: "auto" usa la evidencia de 2.8
    #     opciones: "auto" | "valor_absoluto" (error de signo) | "devolucion" (se netea en la boleta)
    "cantidad_negativa": "auto",
    # 2.9 Columnas redundantes: "auto" conserva la versión que pasa la prueba de campañas; region siempre desde la tienda
    "mes_vs_fecha": "auto",
    # 2.11 Categorías: se aplican las propuestas automáticas
    "categorias": "aplicar_propuesta",
    # 2.3 SKU repetido en el maestro: se conserva el registro activo; si hay empate, el primero
    "sku_repetido": "preferir_activo",
}
# Correcciones que requieren conocimiento externo (detectadas en 2.4). Formato: tabla → (columna clave, valor) → {columna: nuevo valor}
CORRECCIONES_MANUALES = {
    "comunas": {("comuna", "La Florida"): {"region_id": 1}},   # La Florida es una comuna de la Región Metropolitana
}
# Categorías que la detección no pudo resolver (2.11): normalizado → categoría
CATEGORIAS_MANUALES = {}

# Resolución de las decisiones "auto" con la evidencia de la sección 2
if DECISIONES["monto_discrepante"] == "auto":
    dev_precio = abs(comparacion.loc["líneas con discrepancia", "precio_unitario / precio_lista (mediana)"] -
                     comparacion.loc["líneas sin discrepancia", "precio_unitario / precio_lista (mediana)"]) if "líneas con discrepancia" in comparacion.index else 0
    dev_monto = abs(comparacion.loc["líneas con discrepancia", "monto_linea implícito / precio_lista (mediana)"] -
                    comparacion.loc["líneas sin discrepancia", "monto_linea implícito / precio_lista (mediana)"]) if "líneas con discrepancia" in comparacion.index else 0
    DECISIONES["monto_discrepante"] = "recalcular" if dev_monto > dev_precio else "monto_linea"
if DECISIONES["cantidad_negativa"] == "auto":
    DECISIONES["cantidad_negativa"] = "devolucion" if EVIDENCIA_DEVOLUCION else "valor_absoluto"
if DECISIONES["mes_vs_fecha"] == "auto":
    ok_fecha = camp_val["% dentro según fecha"].dropna()
    ok_mes = camp_val["% dentro según columna mes"].dropna()
    DECISIONES["mes_vs_fecha"] = "usar_fecha" if (ok_fecha.mean() if len(ok_fecha) else 100) >= (ok_mes.mean() if len(ok_mes) else 0) else "usar_mes"

print("RESULTADO: decisiones vigentes")
pd.DataFrame(list(DECISIONES.items()), columns=["problema", "decisión"])

### 3.1 Registro de limpieza
**Objetivo:** cada vez que se aplica una decisión, se anota cuántos registros afectó, por qué y qué pasaría si el supuesto fuera falso. El registro se exporta a `outputs/tablas/registro_limpieza.csv` y es la base de la bitácora de supuestos del informe.

In [ ]:
REGISTRO = []

def registrar(fuente, regla, afectados, total, accion, motivo, efecto_si_falso):
    REGISTRO.append({"n": len(REGISTRO) + 1, "fuente": fuente, "regla": regla, "registros_afectados": int(afectados),
                     "total_registros": int(total), "pct_afectado": round(100 * afectados / total, 2) if total else 0.0,
                     "accion": accion, "motivo": motivo, "efecto_si_el_supuesto_fuera_falso": efecto_si_falso})
    print(f"[{len(REGISTRO):02d}] {fuente} · {regla}: {afectados:,} de {total:,} → {accion}")

print("RESULTADO: registro inicializado")

### 3.2 Limpieza de geografía, clientes y maestro
**Objetivo:** aplicar las correcciones manuales de referencias, normalizar el maestro de productos y dejar los clientes con sus fechas convertidas.

In [ ]:
comunas = pos["comunas"].copy()
for (col_clave, valor), cambios in CORRECCIONES_MANUALES.get("comunas", {}).items():
    filtro = comunas[col_clave].astype(str).str.strip() == valor
    for col, nuevo in cambios.items():
        registrar("POS · comunas", f"Corrección manual: {col_clave} = {valor}", filtro.sum(), len(comunas),
                  f"{col} → {nuevo}", "Referencia huérfana detectada en 2.4; se corrige con conocimiento externo.",
                  "Sin corregir, las boletas de esa comuna quedan sin región.")
        comunas.loc[filtro, col] = nuevo
huerf_rest = ~comunas["region_id"].isin(pos["regiones"]["region_id"])
if huerf_rest.any():
    print("Atención: siguen comunas con región inexistente:", comunas.loc[huerf_rest, "comuna"].tolist())

clientes = pos["clientes"].copy()
clientes["fecha_alta"] = fecha_alta.values
clientes = clientes.merge(cli[["cliente_id", "rut_valido"]], on="cliente_id", how="left")
registrar("POS · clientes", "fecha_alta convertida desde varios formatos", len(clientes), len(clientes), "Fecha estándar",
          "Formatos mixtos detectados en 2.1; orden día-mes decidido con evidencia en 2.5.", "—")

m = maestro_raw[~basura_maestro].copy()
registrar("Maestro (CSV)", "Filas cuya clave no sigue el patrón del SKU", basura_maestro.sum(), len(maestro_raw), "Eliminadas",
          "No son productos (detectadas en 2.2).", "—")
for col in ["sku", "nombre", "categoria", "subcategoria", "marca", "proveedor"]:
    m[col] = m[col].astype(str).str.strip()
m["sku"] = m["sku"].str.upper()
m["costo_unitario"] = NUM[("maestro", "costo_unitario")].loc[m.index]
m["precio_lista"] = NUM[("maestro", "precio_lista")].loc[m.index]
VERDADERO, FALSO = {"si", "1", "true", "s", "activo", "yes"}, {"no", "0", "false", "n", "inactivo"}
act = m["activo"].map(normalizar_texto)
desconocido = ~act.isin(VERDADERO | FALSO)
m["activo"] = np.where(act.isin(VERDADERO), True, np.where(act.isin(FALSO), False, None))
registrar("Maestro (CSV)", "Valor de 'activo' vacío o no reconocido", desconocido.sum(), len(m), "Se deja como desconocido (nulo)",
          f"Valores no reconocidos: {sorted(act[desconocido].unique().tolist())}. No se puede inferir el estado.",
          "No afecta la métrica; solo la preferencia entre SKU repetidos.")
mapa_cat = dict(zip(variantes_cat["normalizado"], variantes_cat["propuesta"]))
mapa_cat.update(CATEGORIAS_MANUALES)
cat_antes = m["categoria"].copy()
m["categoria"] = m["categoria"].map(normalizar_texto).map(mapa_cat)
registrar("Maestro (CSV)", "Categoría normalizada a su forma canónica", (cat_antes != m["categoria"]).sum(), len(m),
          f"{cat_antes.nunique()} escrituras → {m['categoria'].nunique()} categorías",
          "Variantes de mayúsculas, tildes y abreviaciones (2.11).", "Solo afecta cortes por categoría.")
if m["categoria"].isna().any():
    print("Atención: categorías sin resolver; agréguenlas a CATEGORIAS_MANUALES:", cat_antes[m["categoria"].isna()].unique())
orden = m.assign(_act=m["activo"].map({True: 0, False: 2}).fillna(1)).sort_values(["sku", "_act"], kind="stable")
rep = orden.duplicated("sku", keep="first")
registrar("Maestro (CSV)", "SKU repetido", rep.sum(), len(m), "Se conserva el registro activo (o el primero)",
          "Una clave debe identificar un solo producto.", "Si el vigente fuera el otro registro, cambia el costo de ese SKU.")
maestro = orden[~rep].drop(columns="_act").sort_values("sku").reset_index(drop=True)
print(f"RESULTADO: maestro limpio con {len(maestro)} productos y {maestro['categoria'].nunique()} categorías; "
      f"{len(comunas)} comunas; {len(clientes)} clientes")

### 3.3 Limpieza de boletas
**Objetivo:** dejar una fila por boleta, con su fecha estándar, dentro del período, y sin las columnas redundantes que resultaron no confiables.

In [ ]:
boletas = pos["boletas"].copy()
boletas["fecha"] = fecha_boleta.values
n_boletas_orig = len(boletas)

dup = boletas.duplicated()
registrar("POS · boletas", "Filas duplicadas exactas", dup.sum(), n_boletas_orig, "Eliminadas (se conserva una)",
          "La misma transacción cargada dos veces (2.3).", "Si fueran compras distintas, la frecuencia estaría levemente subestimada.")
boletas = boletas[~dup]

sin_fecha = boletas["fecha"].isna()
registrar("POS · boletas", "Fecha no interpretable", sin_fecha.sum(), len(boletas), "Eliminadas", "Sin fecha no se ubica en el tiempo.", "—")
boletas = boletas[~sin_fecha]

fuera_p = ~boletas["fecha"].between(PERIODO_INI, PERIODO_FIN)
registrar("POS · boletas", "Fecha fuera del período de análisis", fuera_p.sum(), len(boletas), "Excluidas (guardadas en outputs/calidad)",
          "El caso analiza el año completo; los meses fuera del período tienen un volumen muy inferior al normal (2.5), "
          "así que son registros sueltos y no una extensión de los datos.",
          "Si fueran compras del período con el año mal digitado, se pierde ese porcentaje de boletas.")
boletas = boletas[~fuera_p]

registrar("POS · boletas", "Columna 'mes' no coincide con la fecha", mes_inc.sum(), n_boletas_orig,
          "Columna descartada" if DECISIONES["mes_vs_fecha"] == "usar_fecha" else "Se usa 'mes'",
          "La prueba con campañas (2.9) indica que la fecha es la versión confiable.", "Si 'mes' fuera la correcta, esas boletas cambiarían de mes.")
registrar("POS · boletas", "Columna 'region' no coincide con la tienda", reg_inc.sum(), n_boletas_orig,
          "Columna descartada; la región se toma de tienda → comuna → región",
          "La región es atributo de la tienda; la columna tampoco representa la región del cliente (2.9).", "Solo afecta cortes por región.")
boletas = boletas.drop(columns=["mes", "region"])
print(f"RESULTADO: {len(boletas):,} boletas tras limpiar la cabecera (de {n_boletas_orig:,} originales)")

### 3.4 Limpieza del detalle y monto de cada boleta
**Objetivo:** dejar cada línea con su monto correcto según las decisiones de 2.7 y 2.8, y calcular el monto neto de cada boleta.

In [ ]:
det = det_chk[det_chk["boleta_id"].isin(boletas["boleta_id"])].copy()
registrar("POS · boleta_detalle", "Líneas de boletas excluidas en 3.3", (~det_chk["boleta_id"].isin(boletas["boleta_id"])).sum(),
          len(det_chk), "Eliminadas", "Pertenecen a boletas fuera del período o duplicadas.", "—")
det["sku"] = det["sku_n"]

if DECISIONES["cantidad_negativa"] == "valor_absoluto":
    n_neg = (det["cantidad"] < 0).sum()
    det["cantidad"] = det["cantidad"].abs(); det["monto"] = det["monto"].abs(); det["calculado"] = det["calculado"].abs()
    registrar("POS · boleta_detalle", "Cantidad negativa", n_neg, len(det), "Se toma el valor absoluto (error de signo)",
              "No hay evidencia de devolución: los clientes no habían comprado esos productos antes con más frecuencia que en una "
              "línea cualquiera, el producto no aparece en positivo en la misma boleta y la distribución de cantidades es igual a la de "
              "las líneas positivas (2.8).",
              "Si fueran devoluciones, el monto de esas boletas estaría sobrestimado y algunas no contarían como visita de compra.")
else:
    registrar("POS · boleta_detalle", "Cantidad negativa", (det["cantidad"] < 0).sum(), len(det), "Se conserva: devolución neteada en la boleta",
              "La evidencia de 2.8 es compatible con devoluciones.", "Si fueran errores de signo, el monto estaría subestimado.")

trunc = det["tipo"] == "truncamiento"
registrar("POS · boleta_detalle", "Diferencia menor a $1 por decimales del descuento", trunc.sum(), len(det), "Se conserva monto_linea",
          "El monto es el cálculo sin decimales (truncado); es consistente.", "—")
disc = det["tipo"] == "discrepancia"
if DECISIONES["monto_discrepante"] == "recalcular":
    det.loc[disc, "monto"] = np.trunc(det.loc[disc, "calculado"])
    registrar("POS · boleta_detalle", "monto_linea con discrepancia de $1 o más", disc.sum(), len(det),
              "Monto recalculado = cantidad × precio × (1 − descuento), sin decimales",
              "En esas líneas el precio unitario coincide con el precio de lista del maestro, pero el monto implica un precio mayor (2.7). "
              "Las diferencias no se explican por redondeo en efectivo: superan los $5 y no se concentran en ese medio de pago.",
              "Si el monto original fuera el correcto, las ventas totales cambian en el porcentaje que se informa abajo.")
else:
    registrar("POS · boleta_detalle", "monto_linea con discrepancia de $1 o más", disc.sum(), len(det), "Se conserva monto_linea",
              "Decisión del grupo.", "Si el correcto fuera el cálculo, las ventas cambian en el porcentaje que se informa abajo.")
impacto_monto = 100 * (det_chk.loc[det.index, "monto"].abs().sum() / det["monto"].sum() - 1)

monto_boleta = det.groupby("boleta_id")["monto"].sum()
boletas["monto_nominal"] = boletas["boleta_id"].map(monto_boleta)
no_compra = boletas["monto_nominal"] <= 0
registrar("POS · boletas", "Boleta con monto neto ≤ 0", no_compra.sum(), len(boletas), "Excluida del análisis de actividad",
          "Una boleta sin compra neta no es una visita de compra.", "—")
boletas_validas = boletas[~no_compra].copy()
det = det[det["boleta_id"].isin(boletas_validas["boleta_id"])]
anon = boletas_validas["cliente_id"].isna()
registrar("POS · boletas", "Boleta sin cliente identificado (anónima)", anon.sum(), len(boletas_validas),
          "Conservada para ventas; excluida de la métrica de retención",
          "Sin tarjeta no se puede seguir al cliente en el tiempo.",
          "La gerencia cree que hay clientes frecuentes que no pasan la tarjeta: la retención real podría ser mayor.")
sku_fuera = ~det["sku"].isin(maestro["sku"])
registrar("POS · boleta_detalle", "SKU vendido que no existe en el maestro", sku_fuera.sum(), len(det),
          "Conservado para ventas; sin costo (margen no calculable)", "POS y ERP no comparten la misma fuente de productos (entrevista).",
          "El margen se calcula sobre el resto de las líneas.")
print(f"RESULTADO: {len(boletas_validas):,} boletas válidas y {len(det):,} líneas. "
      f"Usar monto_linea original en vez de la decisión tomada cambiaría las ventas en {impacto_monto:+.2f}%.")

### 3.5 Promociones e IPC
**Objetivo:** normalizar las campañas y convertir la serie del IPC, que viene como **variación mensual en %**, en un índice encadenado para deflactar los montos a pesos de diciembre.

In [ ]:
campanas = campanas_raw.copy()
campanas["descuento_pct"] = campanas["descuento_pct"].map(
    lambda x: (lambda v: v / 100 if v >= 1 else v)(float(str(x).replace("%", "").replace(",", "."))))
campanas["fecha_ini"] = pd.to_datetime(campanas["fecha_ini"])
campanas["fecha_fin"] = pd.to_datetime(campanas["fecha_fin"])
campanas["canales"] = campanas["canales"].map(lambda c: ", ".join(c) if isinstance(c, list) else c)
campana_sku = campana_sku_raw.assign(sku=campana_sku_raw["sku"].str.strip().str.upper())

ipc["indice_base_100"] = 100 * (1 + ipc["valor"] / 100).cumprod()
ipc["factor_a_pesos_dic"] = ipc["indice_base_100"].iloc[-1] / ipc["indice_base_100"]
inflacion_anual = ipc["indice_base_100"].iloc[-1] - 100
registrar("API IPC", f"Serie IPC leída desde: {FUENTE_IPC}", len(ipc), 12, "Variaciones encadenadas en un índice (dic del año anterior = 100)",
          "La API entrega variación mensual, no nivel del índice.",
          f"La empresa deflacta con 3% fijo; la inflación de {ANIO} según la serie fue {inflacion_anual:.2f}% "
          "(el INE informa el valor calculado sobre el índice sin redondear).")
print(f"RESULTADO: inflación acumulada {ANIO} = {inflacion_anual:.2f}%")
ipc[["mes", "valor", "indice_base_100", "factor_a_pesos_dic"]].round(4).T

## 4. Modelo dimensional

**Tabla de hechos principal — `fact_boleta`.** *Una fila es una boleta válida emitida en el período* (sin duplicados, con fecha en el período y monto neto mayor que cero).

**Tabla de hechos de soporte — `fact_cliente_mes` (snapshot periódico).** *Una fila es un cliente identificado al cierre de un mes del período*, desde el mes de su primera compra, con los días desde su última compra y si está activo bajo cada ventana. La métrica se calcula sobre esta tabla.

**Dimensiones:** fecha, cliente, tienda (con formato, canal, comuna, región y macrozona), medio de pago, producto y campaña.

**Decisiones de diseño:**
- El grano es la boleta y no la línea: la pregunta es si el cliente *vuelve*, y una visita es una boleta. El detalle se usa para calcular el monto, el margen y si hubo productos en promoción.
- `segmento` (Clásico, Plus, Black) se sobrescribe en el origen (SCD tipo 1, confirmado por la contraparte): se guarda solo como segmento actual y no se usa para cortar la retención en el tiempo.
- La región se toma de la tienda, no de la boleta (2.9).

### 4.1 Dimensiones
**Objetivo:** construir las dimensiones conformadas.

In [ ]:
fechas = pd.date_range(PERIODO_INI, PERIODO_FIN, freq="D")
NOMBRE_MES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
dim_fecha = pd.DataFrame({"fecha_key": fechas.strftime("%Y%m%d").astype(int), "fecha": fechas, "anio": fechas.year,
                          "mes": fechas.month, "nombre_mes": [NOMBRE_MES[i - 1] for i in fechas.month], "trimestre": fechas.quarter,
                          "dia_semana": fechas.dayofweek + 1, "es_fin_de_semana": fechas.dayofweek >= 5})

MACROZONA = {"Metropolitana": "Metropolitana", "Valparaíso": "Centro sur", "Biobío": "Centro sur", "Araucanía": "Sur", "Los Lagos": "Sur"}
dim_tienda = (pos["tiendas"].merge(comunas, on="comuna_id", how="left").merge(pos["regiones"], on="region_id", how="left")
              .rename(columns={"nombre": "tienda"}))
dim_tienda["formato"] = dim_tienda["formato"].replace({"Hiper": "Híper"})
dim_tienda["canal"] = np.where(dim_tienda["formato"] == "Online", "Web", "Presencial")
dim_tienda["region"] = dim_tienda["region"].fillna("Sin región")
dim_tienda["macrozona"] = dim_tienda["region"].map(MACROZONA).fillna("Sin macrozona")
dim_tienda = dim_tienda[["tienda_id", "tienda", "formato", "canal", "comuna", "region", "macrozona"]]

ident_tmp = boletas_validas.dropna(subset=["cliente_id"])
primera_compra = ident_tmp.groupby("cliente_id")["fecha"].min()
dim_cliente = clientes.merge(comunas[["comuna_id", "comuna", "region_id"]], on="comuna_id", how="left") \
                      .merge(pos["regiones"], on="region_id", how="left")
dim_cliente["primera_compra"] = dim_cliente["cliente_id"].map(primera_compra)
# Inicio de la relación: la fecha más temprana entre el alta y la primera compra (2.10 mostró compras anteriores al alta)
dim_cliente["inicio_relacion"] = dim_cliente[["fecha_alta", "primera_compra"]].min(axis=1)
dim_cliente = dim_cliente.assign(cohorte_inicio=dim_cliente["inicio_relacion"].dt.to_period("M").astype(str),
                                 segmento_actual=dim_cliente["segmento"])[
    ["cliente_id", "fecha_alta", "primera_compra", "inicio_relacion", "cohorte_inicio", "comuna", "region", "segmento_actual", "rut_valido"]]
dim_cliente = pd.concat([dim_cliente, pd.DataFrame([{"cliente_id": -1, "segmento_actual": "Anónimo"}])], ignore_index=True)
dim_medio_pago = pos["medios_pago"].copy()
dim_producto = maestro[["sku", "nombre", "categoria", "subcategoria", "marca", "proveedor", "costo_unitario", "precio_lista", "activo"]].copy()
dim_campana = campanas.copy()
print(f"RESULTADO: dim_fecha {len(dim_fecha)} · dim_tienda {len(dim_tienda)} · dim_cliente {len(dim_cliente)} · "
      f"dim_producto {len(dim_producto)} · dim_campana {len(dim_campana)} · dim_medio_pago {len(dim_medio_pago)}")

### 4.2 Tabla de hechos `fact_boleta`
**Objetivo:** agregar el detalle por boleta (monto, margen y promoción), deflactar a pesos de diciembre y **verificar que se respeta el grano**: exactamente una fila por boleta válida.

In [ ]:
d = det.merge(boletas_validas[["boleta_id", "fecha", "tienda_id"]], on="boleta_id") \
       .merge(dim_producto[["sku", "costo_unitario"]], on="sku", how="left") \
       .merge(dim_tienda[["tienda_id", "canal"]], on="tienda_id")
d["costo_linea"] = d["cantidad"] * d["costo_unitario"]
dp = d.merge(campana_sku, on="sku").merge(campanas, on="campana_id")
dp = dp[(dp["fecha"] >= dp["fecha_ini"].dt.normalize()) & (dp["fecha"] <= dp["fecha_fin"]) &
        np.array([c in cs for c, cs in zip(dp["canal"], dp["canales"])], dtype=bool)]
promo_por_boleta = dp.groupby("boleta_id")["campana_id"].first()

agg = d.groupby("boleta_id").agg(n_lineas=("linea", "size"), unidades=("cantidad", "sum"),
                                 costo_total=("costo_linea", lambda s: s.sum(min_count=1)),
                                 lineas_sin_costo=("costo_unitario", lambda s: s.isna().sum()))
fb = boletas_validas.merge(agg, left_on="boleta_id", right_index=True)
fb["mes_p"] = fb["fecha"].dt.to_period("M")
fb = fb.merge(ipc[["mes", "factor_a_pesos_dic"]], left_on="mes_p", right_on="mes", how="left")
fb["monto_real_dic"] = fb["monto_nominal"] * fb["factor_a_pesos_dic"]
fb["margen_bruto_nominal"] = np.where(fb["lineas_sin_costo"] == 0, fb["monto_nominal"] - fb["costo_total"], np.nan)
fb["margen_bruto_real_dic"] = fb["margen_bruto_nominal"] * fb["factor_a_pesos_dic"]
fb["campana_id"] = fb["boleta_id"].map(promo_por_boleta)
fb["tiene_promo"] = fb["campana_id"].notna()
fb["cliente_id"] = fb["cliente_id"].fillna(-1).astype(int)
fb["fecha_key"] = fb["fecha"].dt.strftime("%Y%m%d").astype(int)
fact_boleta = fb[["boleta_id", "fecha_key", "fecha", "tienda_id", "cliente_id", "medio_id", "campana_id", "n_lineas", "unidades",
                  "monto_nominal", "monto_real_dic", "margen_bruto_nominal", "margen_bruto_real_dic", "tiene_promo"]] \
    .sort_values("fecha").reset_index(drop=True)

assert fact_boleta["boleta_id"].is_unique and len(fact_boleta) == len(boletas_validas), "fact_boleta rompe su grano"
assert fact_boleta["monto_real_dic"].notna().all(), "Hay boletas sin factor IPC (mes faltante en la serie)"
print(f"RESULTADO: fact_boleta con {len(fact_boleta):,} filas (una por boleta válida; grano verificado)")
print(f"  ventas nominales ${fact_boleta['monto_nominal'].sum():,.0f} · reales (pesos de dic) ${fact_boleta['monto_real_dic'].sum():,.0f}")
print(f"  boletas con productos en promoción: {100*fact_boleta['tiene_promo'].mean():.1f}% · margen calculable en "
      f"{100*fact_boleta['margen_bruto_nominal'].notna().mean():.1f}% de las boletas")

### 4.3 Tabla de hechos `fact_cliente_mes`
**Objetivo:** construir el snapshot de actividad de cada cliente identificado al cierre de cada mes y verificar el grano.

In [ ]:
ident = fact_boleta[fact_boleta["cliente_id"] > 0].copy()
cierres = pd.date_range(PERIODO_INI, PERIODO_FIN, freq="ME")
filas = []
for cid, g in ident.groupby("cliente_id"):
    f_c, m_c = g["fecha"].values, g["monto_real_dic"].values
    for cierre in cierres:
        if cierre < f_c.min():
            continue
        hasta = f_c <= np.datetime64(cierre)
        en_mes = hasta & (f_c >= np.datetime64(cierre.replace(day=1)))
        filas.append((cid, cierre, int(en_mes.sum()), float(m_c[en_mes].sum()), int((cierre - pd.Timestamp(f_c[hasta].max())).days)))
fact_cliente_mes = pd.DataFrame(filas, columns=["cliente_id", "fecha_cierre", "compras_mes", "monto_real_mes", "dias_desde_ultima_compra"])
for W in sorted(set(VENTANAS_SENSIBILIDAD) | {W_PRINCIPAL, W_ALTERNATIVA}):
    fact_cliente_mes[f"activo_w{W}"] = fact_cliente_mes["dias_desde_ultima_compra"] < W   # compró en (cierre − W, cierre]
assert not fact_cliente_mes.duplicated(["cliente_id", "fecha_cierre"]).any(), "fact_cliente_mes rompe su grano"

for nombre, df in {"dim_fecha": dim_fecha, "dim_tienda": dim_tienda, "dim_cliente": dim_cliente, "dim_medio_pago": dim_medio_pago,
                   "dim_producto": dim_producto, "dim_campana": dim_campana, "fact_boleta": fact_boleta,
                   "fact_cliente_mes": fact_cliente_mes}.items():
    df.to_csv(f"{PROC}/{nombre}.csv", index=False, encoding="utf-8")
print(f"RESULTADO: fact_cliente_mes con {len(fact_cliente_mes):,} filas (cliente × cierre de mes; grano verificado). "
      f"Modelo guardado en {PROC}/")

## 5. Métrica: retención de la base activa

**Cliente activo en la fecha d con ventana W:** tiene al menos una boleta válida en (d − W, d].

**Métrica principal — Retención de la base activa (RBA):**

$$RBA_W = \frac{|A_W(\text{fecha base}) \cap A_W(\text{cierre})|}{|A_W(\text{fecha base})|}$$

- **Numerador:** clientes activos en la fecha base que siguen activos al cierre del período.
- **Denominador:** clientes activos en la fecha base.
- **Filtros:** clientes identificados; boletas válidas del período.
- **Unidad:** % de clientes.
- **Fecha base:** primer cierre de mes con W_PRINCIPAL días de historia completos dentro de los datos.

**Métrica complementaria — Índice de reposición:** clientes nuevos (inicio de relación después de la fecha base) activos al cierre, dividido por clientes de la base que se perdieron. Mayor que 1: la cadena repone más clientes de los que pierde.

**Decisión de definición 1 (la que no cerró la contraparte):** W = 90 días (principal) frente a 30 días (creencia de la gerencia).
**Decisión de definición 2:** "nuevo" según el inicio de la relación (la fecha más temprana entre alta y primera compra) frente a "primera compra observada en los datos".

### 5.1 Métrica bajo las dos ventanas
**Objetivo:** calcular la RBA y el índice de reposición con la ventana principal y la alternativa.

In [ ]:
def activos(fecha, W, snapshot=None):
    s = fact_cliente_mes if snapshot is None else snapshot
    return set(s.loc[(s["fecha_cierre"] == fecha) & s[f"activo_w{W}"], "cliente_id"])

inicio = dim_cliente.set_index("cliente_id")["inicio_relacion"]
nuevos_inicio = set(inicio[(inicio > FECHA_BASE) & (inicio <= FECHA_CIERRE)].index)
primera_obs = ident.groupby("cliente_id")["fecha"].min()
nuevos_primera_obs = set(primera_obs[primera_obs > FECHA_BASE].index)

def calcular_rba(W, nuevos=nuevos_inicio, snapshot=None):
    A0, A1 = activos(FECHA_BASE, W, snapshot), activos(FECHA_CIERRE, W, snapshot)
    retenidos, perdidos = A0 & A1, A0 - A1
    nuevos_act = (A1 & nuevos) - A0
    return {"ventana_dias": W, "base_activa": len(A0), "retenidos": len(retenidos), "perdidos": len(perdidos),
            "RBA_pct": round(100 * len(retenidos) / len(A0), 1), "activos_cierre": len(A1), "nuevos_activos_cierre": len(nuevos_act),
            "reactivados_u_otros_cierre": len(A1 - retenidos - nuevos_act),
            "pct_cierre_retenidos": round(100 * len(retenidos) / len(A1), 1),
            "pct_cierre_nuevos": round(100 * len(nuevos_act) / len(A1), 1),
            "indice_reposicion": round(len(nuevos_act) / len(perdidos), 2) if perdidos else np.nan}

ficha = pd.DataFrame([calcular_rba(W_PRINCIPAL), calcular_rba(W_ALTERNATIVA)])
ficha.insert(0, "definicion", [f"Principal (W = {W_PRINCIPAL} días)", f"Alternativa (W = {W_ALTERNATIVA} días)"])
ficha.to_csv(f"{TAB}/ficha_metrica.csv", index=False)
rp, ra = ficha.iloc[0], ficha.iloc[1]
misma_direccion = (rp.indice_reposicion > 1) == (ra.indice_reposicion > 1)
print(f"RESULTADO: RBA = {rp.RBA_pct}% con W={W_PRINCIPAL} y {ra.RBA_pct}% con W={W_ALTERNATIVA} "
      f"(diferencia {rp.RBA_pct - ra.RBA_pct:.1f} puntos). Índice de reposición {rp.indice_reposicion} vs {ra.indice_reposicion}.")
print("  La conclusión sobre retener o reponer", "se mantiene" if misma_direccion else "CAMBIA", "entre ambas ventanas.")
ficha.set_index("definicion").T

### 5.2 Sensibilidad a las definiciones y a las decisiones de limpieza
**Objetivo:** mostrar cuánto cambia la respuesta con otras ventanas, con la otra definición de cliente nuevo y con la otra forma de tratar las cantidades negativas.

In [ ]:
sens = pd.DataFrame([calcular_rba(W) for W in VENTANAS_SENSIBILIDAD])
sens.to_csv(f"{TAB}/sensibilidad_ventana.csv", index=False)
sens_nuevo = pd.DataFrame([{"definicion_nuevo": "inicio de relación después de la fecha base (principal)", **calcular_rba(W_PRINCIPAL, nuevos_inicio)},
                           {"definicion_nuevo": "primera compra observada después de la fecha base", **calcular_rba(W_PRINCIPAL, nuevos_primera_obs)}])
sens_nuevo.to_csv(f"{TAB}/sensibilidad_definicion_nuevo.csv", index=False)

# Tratamiento alternativo de negativos: recalcula qué boletas cuentan como visita
alt = "devolucion" if DECISIONES["cantidad_negativa"] == "valor_absoluto" else "valor_absoluto"
det_alt = det_chk[det_chk["boleta_id"].isin(boletas["boleta_id"])].copy()
if DECISIONES["monto_discrepante"] == "recalcular":
    det_alt.loc[det_alt["tipo"] == "discrepancia", "monto"] = np.trunc(det_alt.loc[det_alt["tipo"] == "discrepancia", "calculado"])
det_alt["monto"] = det_alt["monto"].abs() if alt == "valor_absoluto" else det_alt["monto"]
validas_alt = boletas[boletas["boleta_id"].map(det_alt.groupby("boleta_id")["monto"].sum()) > 0].dropna(subset=["cliente_id"])
snap_alt = []
for cid, g in validas_alt.groupby("cliente_id"):
    for cierre in (FECHA_BASE, FECHA_CIERRE):
        hasta = g["fecha"][g["fecha"] <= cierre]
        if len(hasta):
            snap_alt.append((int(cid), cierre, (cierre - hasta.max()).days))
snap_alt = pd.DataFrame(snap_alt, columns=["cliente_id", "fecha_cierre", "dias"])
for W in (W_PRINCIPAL, W_ALTERNATIVA):
    snap_alt[f"activo_w{W}"] = snap_alt["dias"] < W
sens_neg = pd.DataFrame([{"tratamiento_negativos": f"{DECISIONES['cantidad_negativa']} (principal)", **calcular_rba(W_PRINCIPAL)},
                         {"tratamiento_negativos": alt, **calcular_rba(W_PRINCIPAL, snapshot=snap_alt)}])
sens_neg.to_csv(f"{TAB}/sensibilidad_negativos.csv", index=False)

print("RESULTADO:")
print(" Ventana →\n", sens[["ventana_dias", "base_activa", "RBA_pct", "pct_cierre_nuevos", "indice_reposicion"]].to_string(index=False))
print("\n Definición de nuevo →\n", sens_nuevo[["definicion_nuevo", "nuevos_activos_cierre", "pct_cierre_nuevos", "indice_reposicion"]].to_string(index=False))
print("\n Tratamiento de negativos →\n", sens_neg[["tratamiento_negativos", "base_activa", "RBA_pct", "indice_reposicion"]].to_string(index=False))

### 5.3 Intervalo entre compras
**Objetivo:** medir cada cuánto vuelve a comprar un cliente. Es el dato que justifica la ventana: una ventana más corta que el ciclo normal de recompra clasifica como perdidos a clientes que simplemente no han vuelto todavía.

In [ ]:
ident = ident.sort_values(["cliente_id", "fecha"])
ident["dias_desde_anterior"] = ident.groupby("cliente_id")["fecha"].diff().dt.days
gaps = ident["dias_desde_anterior"].dropna()
pct_gap = {W: round(100 * (gaps <= W).mean(), 1) for W in sorted(set(VENTANAS_SENSIBILIDAD))}
print(f"RESULTADO: {len(gaps):,} recompras · mediana {gaps.median():.0f} días · percentil 90 {gaps.quantile(.9):.0f} días")
print(" % de recompras que ocurren dentro de W días:", pct_gap)

### 5.4 Retención por formato de la tienda principal
**Objetivo:** responder lo que pidió la gerencia en la entrevista: una tasa de retención por local o tipo de local, no solo una general. La tienda principal de cada cliente es la que más usó antes de la fecha base.

In [ ]:
previo = ident[ident["fecha"] <= FECHA_BASE].merge(dim_tienda, on="tienda_id")
formato_principal = previo.groupby("cliente_id")["formato"].agg(lambda s: s.value_counts().index[0])
por_formato = []
for W in (W_PRINCIPAL, W_ALTERNATIVA):
    A0, A1 = activos(FECHA_BASE, W), activos(FECHA_CIERRE, W)
    fp = formato_principal[formato_principal.index.isin(A0)]
    for fmt, ids in fp.groupby(fp).groups.items():
        ids = set(ids)
        por_formato.append({"ventana_dias": W, "formato": fmt, "base_activa": len(ids), "RBA_pct": round(100 * len(ids & A1) / len(ids), 1)})
por_formato = pd.DataFrame(por_formato)
por_formato.to_csv(f"{TAB}/retencion_por_formato.csv", index=False)
print("RESULTADO:")
por_formato.pivot(index="formato", columns="ventana_dias", values=["base_activa", "RBA_pct"])

### 5.5 Recompra de clientes nuevos según si su primera compra tuvo promoción
**Objetivo:** usar la fuente de promociones para ver si los clientes captados con promoción vuelven igual que el resto. Solo se consideran clientes con al menos W_PRINCIPAL días observables después de su primera compra.

In [ ]:
primera_b = ident.sort_values("fecha").drop_duplicates("cliente_id").set_index("cliente_id")[["fecha", "tiene_promo"]]
primera_b = primera_b[primera_b.index.isin(nuevos_inicio | set(inicio[inicio >= PERIODO_INI].index))]
primera_b = primera_b[primera_b["fecha"] <= FECHA_CIERRE - pd.Timedelta(days=W_PRINCIPAL)]
segunda = ident.sort_values("fecha").groupby("cliente_id")["fecha"].nth(1)
segunda = pd.Series(segunda.values, index=ident.loc[segunda.index, "cliente_id"].values)
primera_b["volvio_dentro_de_W"] = (segunda.reindex(primera_b.index) - primera_b["fecha"]).dt.days <= W_PRINCIPAL
promo_tab = primera_b.groupby("tiene_promo")["volvio_dentro_de_W"].agg(clientes="size", pct_volvio="mean").reset_index()
promo_tab["pct_volvio"] = (100 * promo_tab["pct_volvio"]).round(1)
promo_tab["primera_compra_con_promo"] = promo_tab["tiene_promo"].map({True: "Sí", False: "No"})
promo_tab = promo_tab[["primera_compra_con_promo", "clientes", "pct_volvio"]]
promo_tab.to_csv(f"{TAB}/recompra_segun_promo.csv", index=False)
print("RESULTADO: (muestras chicas deben leerse como señal, no como conclusión)")
promo_tab

## 6. Caso de negocio en pesos reales
**Objetivo:** traducir la pérdida de clientes a pesos. Para cada cliente perdido (activo en la fecha base, inactivo al cierre) se toma su gasto mensual promedio antes de la fecha base, en pesos de diciembre, y se proyecta a los meses restantes del período. Es lo que habrían comprado si hubieran seguido al mismo ritmo. El margen se calcula con el costo unitario del maestro.

**Supuestos** (van a la bitácora): el ritmo de compra antes de la fecha base se habría mantenido; el costo unitario del maestro es el vigente todo el año; no se descuenta a los clientes que se habrían perdido de todas formas.

In [ ]:
meses_base = FECHA_BASE.month - PERIODO_INI.month + 1
meses_resto = 12 - meses_base
gasto_previo = ident[ident["fecha"] <= FECHA_BASE].groupby("cliente_id")["monto_real_dic"].sum()
margen_pct = fact_boleta["margen_bruto_nominal"].sum() / fact_boleta.loc[fact_boleta["margen_bruto_nominal"].notna(), "monto_nominal"].sum()
caso = []
for W in (W_PRINCIPAL, W_ALTERNATIVA):
    perdidos = list(activos(FECHA_BASE, W) - activos(FECHA_CIERRE, W))
    g = gasto_previo.reindex(perdidos).fillna(0)
    ventas = g.sum() / meses_base * meses_resto
    caso.append({"ventana_dias": W, "clientes_perdidos": len(perdidos), "gasto_mensual_promedio_previo": round(g.mean() / meses_base),
                 "ventas_no_capturadas": round(ventas), "margen_bruto_no_capturado": round(ventas * margen_pct)})
caso = pd.DataFrame(caso)
caso.to_csv(f"{TAB}/caso_negocio.csv", index=False)
frecuencia = ident.groupby("cliente_id").agg(boletas=("boleta_id", "size"), gasto=("monto_real_dic", "sum"))
gasto_rec = frecuencia.loc[frecuencia["boletas"] > 1, "gasto"].mean()
gasto_unica = frecuencia.loc[frecuencia["boletas"] == 1, "gasto"].mean()
print(f"RESULTADO: margen bruto promedio {100*margen_pct:.1f}% · meses proyectados {meses_resto}")
print(f"  gasto anual real de un cliente recurrente ${gasto_rec:,.0f} vs comprador de una sola vez ${gasto_unica:,.0f} (×{gasto_rec/gasto_unica:.0f})")
caso

## 7. Cobertura de datos
**Objetivo:** informar qué porcentaje de las boletas originales quedó fuera de la métrica y por qué, en un solo número.

In [ ]:
n_dup = int(dup.sum()); n_sinf = int(sin_fecha.sum()); n_fuera = int(fuera_p.sum())
n_nocompra = int(no_compra.sum()); n_anon = int(anon.sum())
n_en_metrica = len(ident)
cobertura = pd.DataFrame([{"motivo": "Duplicadas", "boletas": n_dup}, {"motivo": "Fecha no interpretable", "boletas": n_sinf},
                          {"motivo": "Fuera del período", "boletas": n_fuera}, {"motivo": "Monto neto ≤ 0", "boletas": n_nocompra},
                          {"motivo": "Anónimas", "boletas": n_anon}])
assert n_boletas_orig - n_en_metrica == cobertura["boletas"].sum(), "La cobertura no cuadra"
cobertura = pd.concat([cobertura, pd.DataFrame([{"motivo": "TOTAL excluido de la métrica", "boletas": n_boletas_orig - n_en_metrica},
                                                {"motivo": "Usadas en la métrica", "boletas": n_en_metrica}])], ignore_index=True)
cobertura["pct_sobre_originales"] = (100 * cobertura["boletas"] / n_boletas_orig).round(1)
cobertura.to_csv(f"{TAB}/cobertura.csv", index=False)
pct_excluido = cobertura.iloc[-2]["pct_sobre_originales"]
pct_ventas_ident = ident["monto_nominal"].sum() / fact_boleta["monto_nominal"].sum()
registro_limpieza = pd.DataFrame(REGISTRO)
registro_limpieza.to_csv(f"{TAB}/registro_limpieza.csv", index=False, encoding="utf-8-sig")
print(f"RESULTADO: {pct_excluido}% de las boletas originales quedó fuera de la métrica; "
      f"las identificadas representan el {100*pct_ventas_ident:.1f}% de las ventas válidas.")
cobertura

## 8. Visualizaciones
**Objetivo:** producir entre 3 y 5 gráficos que respondan la pregunta, cada uno con su unidad en el título y su fuente al pie. Se guardan en `outputs/figuras/` a 200 dpi.

In [ ]:
AZUL, NARANJA, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
GRIS, TINTA, TINTA2, GRILLA = "#9a9893", "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({"font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "axes.edgecolor": GRILLA, "axes.labelcolor": TINTA2, "xtick.color": TINTA2, "ytick.color": TINTA2,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRILLA,
                     "grid.linewidth": 0.6, "axes.axisbelow": True, "legend.frameon": False, "figure.dpi": 110})

def guardar(fig, nombre, fuente=FUENTE_PIE):
    fig.text(0.01, 0.005, fuente, fontsize=7.5, color=TINTA2, ha="left", va="bottom")
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(f"{FIG}/{nombre}.png", dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()

def miles(x):
    return f"{x:,.0f}".replace(",", ".")

print("RESULTADO: estilo de gráficos configurado")

**G1 — Composición de la base activa al cierre según su origen, bajo cada ventana.** Responde directamente la pregunta: cuántos de los clientes activos al cierre vienen de la base retenida y cuántos son nuevos.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.1))
partes = [("retenidos", f"Retenidos de la base al {FECHA_BASE:%d-%m}", AZUL), ("nuevos_activos_cierre", "Nuevos", NARANJA),
          ("reactivados_u_otros_cierre", "Reactivados y otros", GRIS)]
for i, f in enumerate([rp, ra]):
    izq = 0
    for col, lab, colr in partes:
        v = f[col]
        ax.barh(i, v, left=izq, color=colr, height=0.55, edgecolor="white", linewidth=2, label=lab if i == 0 else None)
        if v > 25:
            ax.text(izq + v / 2, i, miles(v), ha="center", va="center", color="white", fontsize=9, fontweight="bold")
        izq += v
    ax.text(izq + 8, i, f"Retención de la base: {f['RBA_pct']:.0f}%", va="center", color=TINTA, fontsize=9)
ax.set_yticks([0, 1], [f"W = {W_PRINCIPAL} días\n(principal)", f"W = {W_ALTERNATIVA} días\n(alternativa)"])
ax.invert_yaxis()
ax.set_xlim(0, max(rp.activos_cierre, ra.activos_cierre) * 1.42)
ax.grid(axis="y", visible=False)
ax.set_xlabel(f"Clientes identificados activos al {FECHA_CIERRE:%d-%m-%Y}")
ax.set_title(f"Clientes activos al cierre de {ANIO} según su origen (n° de clientes)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=3, fontsize=9)
guardar(fig, "G1_base_activa_cierre_por_origen")

**G2 — Clientes que compran cada mes, por estado.** Muestra el flujo: cuántos de los que compran cada mes son recurrentes, nuevos o reactivados.

In [ ]:
ident["mes"] = ident["fecha"].dt.month
prim_mes = ident.drop_duplicates(["cliente_id", "mes"], keep="first").copy()   # primera boleta de cada cliente en cada mes
prim_mes["inicio"] = prim_mes["cliente_id"].map(inicio)

def estado(r, W=W_PRINCIPAL):
    if pd.isna(r["dias_desde_anterior"]):
        return "Nuevo" if r["inicio"] >= PERIODO_INI else "Existente, primera compra observada"
    return "Recurrente" if r["dias_desde_anterior"] <= W else "Reactivado"

prim_mes["estado"] = prim_mes.apply(estado, axis=1)
orden_est = [("Recurrente", AZUL), ("Nuevo", NARANJA), ("Reactivado", AQUA), ("Existente, primera compra observada", GRIS)]
flujo = prim_mes.pivot_table(index="mes", columns="estado", values="cliente_id", aggfunc="count", fill_value=0) \
                .reindex(columns=[o for o, _ in orden_est], fill_value=0).reindex(range(1, 13), fill_value=0)
flujo.to_csv(f"{TAB}/flujo_mensual_estados_w{W_PRINCIPAL}.csv")
fig, ax = plt.subplots(figsize=(8.5, 4))
base = np.zeros(len(flujo))
for col, colr in orden_est:
    ax.bar(flujo.index, flujo[col], bottom=base, color=colr, width=0.72, edgecolor="white", linewidth=1.5, label=col)
    base += flujo[col].values
for x, tot in zip(flujo.index, base):
    ax.text(x, tot + 5, miles(tot), ha="center", fontsize=8, color=TINTA2)
ax.set_xticks(range(1, 13), NOMBRE_MES)
ax.set_ylabel("Clientes identificados que compran en el mes")
ax.set_title(f"Clientes que compran cada mes, por estado (n° de clientes, ventana {W_PRINCIPAL} días)")
ax.legend(loc="upper left", fontsize=8.5, ncol=2)
ax.set_ylim(0, base.max() * 1.3)
ax.grid(axis="x", visible=False)
guardar(fig, "G2_flujo_mensual_por_estado")

**G3 — Días entre compras consecutivas.** Justifica la ventana elegida.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
tope = 180
ax.hist(gaps.clip(upper=tope), bins=np.arange(0, tope + 8, 7), weights=np.ones(len(gaps)) * 100 / len(gaps),
        color=AZUL, edgecolor="white", linewidth=1.2)
for W, colr in ((W_ALTERNATIVA, NARANJA), (W_PRINCIPAL, TINTA)):
    ax.axvline(W, color=colr, linewidth=1.6, linestyle="--")
    ax.text(W + 2, ax.get_ylim()[1] * 0.92, f"{W} días: {pct_gap.get(W, round(100*(gaps<=W).mean(),1)):.0f}% de las\nrecompras ocurre antes",
            color=colr, fontsize=8.5, va="top")
ax.set_xlabel(f"Días entre una compra y la siguiente del mismo cliente (último tramo agrupa ≥ {tope})")
ax.set_ylabel("% de las recompras")
ax.set_title("Intervalo entre compras consecutivas (% de recompras por tramo de 7 días)")
ax.grid(axis="x", visible=False)
guardar(fig, "G3_intervalo_entre_compras")

**G4 — Retención por formato de la tienda principal.** Responde la petición de la gerencia de ver la retención por tipo de local.

In [ ]:
pf = por_formato.pivot(index="formato", columns="ventana_dias", values="RBA_pct")
nb = por_formato[por_formato["ventana_dias"] == W_PRINCIPAL].set_index("formato")["base_activa"]
pf = pf.loc[pf[W_PRINCIPAL].sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(pf))
ax.bar(x - 0.19, pf[W_PRINCIPAL], width=0.36, color=AZUL, label=f"W = {W_PRINCIPAL} días (principal)")
ax.bar(x + 0.19, pf[W_ALTERNATIVA], width=0.36, color=NARANJA, label=f"W = {W_ALTERNATIVA} días (alternativa)")
for xi, (v1, v2) in enumerate(zip(pf[W_PRINCIPAL], pf[W_ALTERNATIVA])):
    ax.text(xi - 0.19, v1 + 1, f"{v1:.0f}%", ha="center", fontsize=8.5, color=TINTA)
    ax.text(xi + 0.19, v2 + 1, f"{v2:.0f}%", ha="center", fontsize=8.5, color=TINTA)
ax.set_xticks(x, [f"{f}\n(base W{W_PRINCIPAL}: {nb[f]} clientes)" for f in pf.index])
ax.set_ylabel(f"% de la base activa al {FECHA_BASE:%d-%m} activa al cierre")
ax.set_ylim(0, pf.max().max() * 1.25)
ax.set_title("Retención de la base activa por formato de la tienda principal (% de clientes)")
ax.legend(loc="upper right", fontsize=8.5)
ax.grid(axis="x", visible=False)
guardar(fig, "G4_retencion_por_formato")

**G5 — Sensibilidad de la respuesta a la ventana.** Muestra cuánto cambia el número según la definición, y que la dirección de la conclusión no cambia.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(sens["ventana_dias"], sens["RBA_pct"], color=AZUL, linewidth=2, marker="o", markersize=7, label="Retención de la base (RBA)")
ax.plot(sens["ventana_dias"], sens["pct_cierre_nuevos"], color=NARANJA, linewidth=2, marker="o", markersize=7,
        label="% de la base al cierre que es nueva")
for _, r in sens.iterrows():
    ax.text(r.ventana_dias, r.RBA_pct - 6, f"{r.RBA_pct:.0f}%", ha="center", fontsize=8.5, color=TINTA)
    ax.text(r.ventana_dias, r.pct_cierre_nuevos + 3, f"{r.pct_cierre_nuevos:.0f}%", ha="center", fontsize=8.5, color=TINTA)
ax.axvspan(W_PRINCIPAL - 4, W_PRINCIPAL + 4, color=GRILLA, zorder=0)
ax.set_xticks(VENTANAS_SENSIBILIDAD, [f"{w} días" for w in VENTANAS_SENSIBILIDAD])
ax.set_ylim(0, 100)
ax.set_xlabel("Ventana de actividad W (días sin comprar para considerar a un cliente perdido)")
ax.set_ylabel("% de clientes")
ax.set_title("Cómo cambia la respuesta según la ventana elegida (% de clientes)")
ax.legend(loc="center right", fontsize=8.5)
ax.grid(axis="x", visible=False)
guardar(fig, "G5_sensibilidad_ventana")

**Diagrama del modelo dimensional.** Se genera desde el código para que coincida con las tablas construidas en la sección 4. Se guarda en `outputs/figuras/`.

In [ ]:
def caja(ax, x, y, titulo, campos, ancho=2.5, hecho=False):
    alto = 0.42 + 0.27 * len(campos)
    ax.add_patch(matplotlib.patches.FancyBboxPatch((x - ancho / 2, y - alto / 2), ancho, alto,
                 boxstyle="round,pad=0.04,rounding_size=0.12", facecolor="#fdf0ea" if hecho else "#eef4fc",
                 edgecolor=NARANJA if hecho else AZUL, linewidth=1.6 if hecho else 1.1))
    ax.text(x, y + alto / 2 - 0.25, titulo, ha="center", va="center", fontsize=9.5, fontweight="bold", color=TINTA)
    for i, c in enumerate(campos):
        ax.text(x - ancho / 2 + 0.14, y + alto / 2 - 0.55 - 0.27 * i, c, fontsize=7.6, color=TINTA2, va="center")
    return (x, y)

fig, ax = plt.subplots(figsize=(11, 7.2))
ax.set_xlim(0, 13); ax.set_ylim(0, 9); ax.axis("off")
fb_c = caja(ax, 5.2, 4.6, "fact_boleta", ["boleta_id (degenerada)", "fecha_key  → dim_fecha", "tienda_id  → dim_tienda",
            "cliente_id → dim_cliente", "medio_id   → dim_medio_pago", "campana_id → dim_campana", "n_lineas, unidades",
            "monto_nominal, monto_real_dic", "margen_bruto (nominal / real)", "tiene_promo"], ancho=3.2, hecho=True)
fcm_c = caja(ax, 10.6, 4.6, "fact_cliente_mes (snapshot)", ["cliente_id   → dim_cliente", "fecha_cierre → dim_fecha",
             "compras_mes, monto_real_mes", "dias_desde_ultima_compra", "activo_w30 … activo_w90"], ancho=3.3, hecho=True)
dims = {
    "dim_fecha": (caja(ax, 10.2, 1.2, "dim_fecha", ["fecha_key", "fecha, mes, trimestre", "dia_semana, fin_de_semana"]), [fb_c, fcm_c]),
    "dim_tienda": (caja(ax, 1.6, 4.6, "dim_tienda", ["tienda_id", "tienda, formato, canal", "comuna, región, macrozona"]), [fb_c]),
    "dim_medio_pago": (caja(ax, 1.6, 7.6, "dim_medio_pago", ["medio_id", "medio"]), [fb_c]),
    "dim_cliente": (caja(ax, 7.9, 8.0, "dim_cliente", ["cliente_id (−1 = anónimo)", "fecha_alta, inicio_relacion",
                    "comuna, región", "segmento_actual (SCD1)"]), [fb_c, fcm_c]),
    "dim_campana": (caja(ax, 5.2, 0.9, "dim_campana", ["campana_id, nombre, tipo", "fecha_ini, fecha_fin, descuento"], ancho=3.0), [fb_c]),
    "dim_producto": (caja(ax, 1.6, 1.3, "dim_producto", ["sku, categoría, marca", "costo_unitario, precio_lista"], ancho=2.6), []),
}
for _, ((dx, dy), destinos) in dims.items():
    for (fx, fy) in destinos:
        ax.plot([dx, fx], [dy, fy], color=GRIS, linewidth=1, zorder=0)
ax.annotate("", xy=(3.55, 3.1), xytext=(2.4, 2.05), arrowprops=dict(arrowstyle="->", color=GRIS, linestyle="--"))
ax.text(3.35, 2.25, "aporta costo y SKU en\npromoción (se agrega\npor boleta)", fontsize=7.5, color=TINTA2, ha="left")
ax.set_title("Modelo dimensional — Multitienda del Sur (Entrega 1)", fontsize=13)
fig.text(0.01, 0.01, "Grano fact_boleta: una fila es una boleta válida emitida en el período.   "
         "Grano fact_cliente_mes: una fila es un cliente identificado al cierre de un mes del período.", fontsize=8.5, color=TINTA)
fig.savefig(f"{FIG}/modelo_dimensional.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

## 9. Resumen de cifras del informe
**Objetivo:** reunir todas las cifras que cita el informe. Si el informe dice otra cosa, el informe está mal.

In [ ]:
cifras = {
    "Boletas originales": n_boletas_orig,
    "Boletas válidas del período": len(fact_boleta),
    "Boletas usadas en la métrica (identificadas)": n_en_metrica,
    "% de boletas originales excluidas de la métrica": pct_excluido,
    "% de boletas válidas anónimas": round(100 * n_anon / len(boletas_validas), 1),
    "% de ventas válidas de clientes identificados": round(100 * pct_ventas_ident, 1),
    "Problemas de calidad detectados": len(resumen_calidad),
    "Fuente IPC usada": FUENTE_IPC,
    f"Inflación acumulada {ANIO} (%)": round(inflacion_anual, 2),
    "Decisión monto discrepante": DECISIONES["monto_discrepante"],
    "Decisión cantidad negativa": DECISIONES["cantidad_negativa"],
    "Cambio en ventas si se usara monto_linea original (%)": round(impacto_monto, 2),
    "Mediana días entre compras": gaps.median(),
    f"% recompras dentro de {W_ALTERNATIVA} / {W_PRINCIPAL} días": f"{pct_gap.get(W_ALTERNATIVA)} / {pct_gap.get(W_PRINCIPAL)}",
    f"RBA W={W_PRINCIPAL} (%)": rp.RBA_pct,
    f"RBA W={W_ALTERNATIVA} (%)": ra.RBA_pct,
    f"Base activa W={W_PRINCIPAL}": rp.base_activa,
    f"Clientes perdidos W={W_PRINCIPAL}": rp.perdidos,
    f"Nuevos activos al cierre W={W_PRINCIPAL}": rp.nuevos_activos_cierre,
    f"% de la base al cierre que es nueva W={W_PRINCIPAL}": rp.pct_cierre_nuevos,
    f"Índice de reposición W={W_PRINCIPAL}": rp.indice_reposicion,
    f"Índice de reposición W={W_ALTERNATIVA}": ra.indice_reposicion,
    "Margen bruto promedio (%)": round(100 * margen_pct, 1),
    f"Ventas no capturadas W={W_PRINCIPAL} ($ dic)": int(caso.iloc[0]["ventas_no_capturadas"]),
    f"Margen no capturado W={W_PRINCIPAL} ($ dic)": int(caso.iloc[0]["margen_bruto_no_capturado"]),
    "Gasto anual real cliente recurrente ($)": round(gasto_rec),
    "Gasto anual real comprador de una vez ($)": round(gasto_unica),
}
cifras_df = pd.DataFrame(list(cifras.items()), columns=["cifra", "valor"])
cifras_df.to_csv(f"{TAB}/cifras_informe.csv", index=False)
print(f"RESULTADO: ejecución completa en {time.time() - T0:.0f} s. Cifras en {TAB}/cifras_informe.csv")
cifras_df